# SYMPHONY results explorer

This notebook loads a SYMPHONY model output bundle, validates the file structure,
and provides an interactive clinical dashboard for patient-level exploration.
All values trace to specific source files within the bundle. Run all cells, then
click **Run SYMPHONY explorer** in section E to begin.

In [ ]:
%%capture _setup_log
# @title B. Setup and configuration 

# --- Setup and imports ---
# Install only what Colab doesn't ship by default
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])

import os
import re
import json
import zipfile
import warnings
from io import BytesIO
from pathlib import Path, PurePosixPath
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple, Union
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML
import yaml

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.figsize": (8, 3.5),
})



In [ ]:
%%capture _ingestion_log
# @title C. Data access and validation 

# --- Ingestion and validation ---

# ============================================================================
# 1) Acquire the zip: Google Drive → ZIP_PATH → Colab upload (in that order)
# ============================================================================
GDRIVE_FILE_ID = os.environ.get("SYMPHONY_BUNDLE_GDRIVE_ID", "")  # opt-in: set to your own approved bundle
ZIP_PATH: Optional[str] = None  # Set manually if you have a local path

_zip_bytes: Optional[bytes] = None

# Strategy A: Google Drive auto-download
if _zip_bytes is None and GDRIVE_FILE_ID:
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "gdown"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        import gdown
        _gdrive_url = f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}"
        _tmp_path = "/tmp/SYMPHONY_results.zip"
        gdown.download(_gdrive_url, _tmp_path, quiet=True, fuzzy=True)
        if os.path.exists(_tmp_path) and os.path.getsize(_tmp_path) > 1000:
            _zip_bytes = Path(_tmp_path).read_bytes()
            print(f"Loaded from Google Drive ({len(_zip_bytes)/1e6:.1f} MB)")
    except Exception as e:
        print(f"Google Drive download skipped: {e}")

# Strategy B: explicit ZIP_PATH
if _zip_bytes is None and ZIP_PATH and Path(ZIP_PATH).exists():
    _zip_bytes = Path(ZIP_PATH).read_bytes()
    print(f"Loaded from ZIP_PATH: {ZIP_PATH} ({len(_zip_bytes)/1e6:.1f} MB)")

# Strategy C: Colab upload
if _zip_bytes is None:
    try:
        from google.colab import files as _colab_files
        print("No zip found. Upload SYMPHONY_results.zip:")
        _uploaded = _colab_files.upload()
        if _uploaded:
            _fname = list(_uploaded.keys())[0]
            _zip_bytes = _uploaded[_fname]
            print(f"Uploaded: {_fname} ({len(_zip_bytes)/1e6:.1f} MB)")
    except ImportError:
        pass

if _zip_bytes is None:
    raise FileNotFoundError(
        "No zip file acquired. Set GDRIVE_FILE_ID, ZIP_PATH, or upload manually."
    )

# ============================================================================
# 2) Open zip and detect root folder
# ============================================================================
ZF = zipfile.ZipFile(BytesIO(_zip_bytes))
_all_names = [n for n in ZF.namelist() if not n.startswith("__MACOSX")]

# Detect root prefix (e.g. "symphony_hpc_v2_fixed/")
_top_dirs = sorted({PurePosixPath(n).parts[0] for n in _all_names if PurePosixPath(n).parts})
if len(_top_dirs) == 1:
    ROOT_PREFIX = _top_dirs[0] + "/"
elif "symphony_hpc_v2_fixed" in _top_dirs:
    ROOT_PREFIX = "symphony_hpc_v2_fixed/"
else:
    ROOT_PREFIX = ""
print(f"Root prefix: '{ROOT_PREFIX}'")

def zpath(rel: str) -> str:
    """Resolve a relative path to full zip path."""
    return ROOT_PREFIX + rel

def zexists(rel: str) -> bool:
    return zpath(rel) in ZF.namelist()

def zread_csv(rel: str) -> pd.DataFrame:
    return pd.read_csv(BytesIO(ZF.read(zpath(rel))))

def zread_json(rel: str) -> Any:
    return json.loads(ZF.read(zpath(rel)))

def zread_text(rel: str) -> str:
    return ZF.read(zpath(rel)).decode("utf-8")

# ============================================================================
# 3) Detect folds (not hardcoded)
# ============================================================================
_fold_pattern = re.compile(r"fold_(\d+)")
_detected_folds: set = set()
for name in _all_names:
    m = _fold_pattern.search(name)
    if m:
        _detected_folds.add(int(m.group(1)))

FOLDS = sorted(_detected_folds)
FOLD_IDS = [f"fold_{f:02d}" for f in FOLDS]
print(f"Detected {len(FOLDS)} folds: {FOLD_IDS}")

if not FOLDS:
    raise ValueError("No fold directories detected in zip.")

# ============================================================================
# 4) Define expected structure and validate
# ============================================================================
ROOT_FILES = [
    "README.md", "pipeline_log.txt", "config_snapshot.yaml",
    "data_dictionary.md", "metadata.json", "metadata_fold_train_stats.json",
]
PER_FOLD_FILES = {
    "02_preprocessing": ["standardisation_stats_train.csv"],
    "03_inference": [
        "global_parameter_summary.csv", "convergence_diagnostics.json",
        "posterior_contraction_ratios.csv", "pathway_stability.csv",
        "L_cond_selected.csv", "L_cond_subsample_indices.csv",
        "tau_conf_selected.csv",
    ],
    "04_forecasting": [
        "forecasts_test.csv", "conditioning_diagnostics_test.csv",
        "threshold_event_probabilities_test.csv",
    ],
    "05_evaluation": [
        "participant_level_metrics.csv", "aggregate_metrics.csv",
        "threshold_event_calibration.csv", "threshold_event_reliability_bins.csv",
        "ppc_summary.json", "suppression_summary_test.csv",
    ],
    "06_translational": [
        "latent_state_trajectories_test.csv", "risk_curves_ui.csv",
        "indices_ui.csv",
    ],
}
SHARED_FILES = {
    "01_splits": ["fold_assignments.csv"],
    "02_preprocessing": [
        "visit_summaries.csv", "endpoint_completeness.csv",
        "boundary_prevalence.csv", "omics_selection_report.csv",
    ],
}

# Build manifest
manifest_rows: List[Dict[str, Any]] = []

def _add_manifest(rel_path: str, required: bool = True) -> None:
    exists = zexists(rel_path)
    info = ZF.getinfo(zpath(rel_path)) if exists else None
    size = info.file_size if info else 0
    schema = ""
    if exists:
        ext = Path(rel_path).suffix.lower()
        try:
            if ext == ".csv":
                df = pd.read_csv(BytesIO(ZF.read(zpath(rel_path))), nrows=0)
                nrows = sum(1 for _ in BytesIO(ZF.read(zpath(rel_path)))) - 1
                schema = f"{nrows} rows, cols: {', '.join(df.columns[:6])}"
                if len(df.columns) > 6:
                    schema += f" (+{len(df.columns)-6} more)"
            elif ext == ".json":
                d = json.loads(ZF.read(zpath(rel_path)))
                keys = list(d.keys())[:6] if isinstance(d, dict) else [f"array[{len(d)}]"]
                schema = f"keys: {', '.join(str(k) for k in keys)}"
            elif ext in (".yaml", ".yml"):
                d = yaml.safe_load(ZF.read(zpath(rel_path)))
                keys = list(d.keys())[:6] if isinstance(d, dict) else ["(not a mapping)"]
                schema = f"keys: {', '.join(str(k) for k in keys)}"
            elif ext == ".md":
                text = ZF.read(zpath(rel_path)).decode("utf-8")
                schema = f"{len(text)} chars, {text.count(chr(10))} lines"
        except Exception as e:
            schema = f"parse error: {e}"
    manifest_rows.append({
        "path": rel_path,
        "exists": exists,
        "size_bytes": size,
        "schema": schema,
        "required": required,
    })

# Root files
for f in ROOT_FILES:
    _add_manifest(f, required=(f == "metadata.json"))

# Shared (non-fold) files
for section, files in SHARED_FILES.items():
    for f in files:
        _add_manifest(f"{section}/{f}", required=True)

# Per-fold files
for fold_id in FOLD_IDS:
    for section, files in PER_FOLD_FILES.items():
        for f in files:
            _add_manifest(f"{section}/{fold_id}/{f}", required=True)

# 07_reviewer_audit (optional)
_audit_files = [n.replace(ROOT_PREFIX, "") for n in _all_names
                if "07_reviewer_audit" in n and not n.endswith("/")]
for af in _audit_files:
    _add_manifest(af, required=False)

MANIFEST = pd.DataFrame(manifest_rows)

# ============================================================================
# 5) Fail fast on missing required files
# ============================================================================
_missing = MANIFEST[(MANIFEST["required"]) & (~MANIFEST["exists"])]
if not _missing.empty:
    print("MISSING REQUIRED FILES:")
    for _, row in _missing.iterrows():
        print(f"  - {row['path']}")
    raise FileNotFoundError(
        f"{len(_missing)} required file(s) missing from zip. See list above."
    )

# ============================================================================
# 6) Report
# ============================================================================
_n_exist = MANIFEST["exists"].sum()
_n_total = len(MANIFEST)
_n_required = MANIFEST["required"].sum()
print(f"Manifest: {_n_exist}/{_n_total} files found ({_n_required} required, all present)")

# Show compact summary
_MANIFEST_STYLED = (
    MANIFEST[["path", "exists", "size_bytes", "schema"]]
    .style.set_caption("Bundle manifest (truncated schema)")
    .hide(axis="index")
)

In [ ]:
%%capture _err_log
# @title C2. Error gate 
try:
    _ = ZF, ROOT_PREFIX, FOLD_IDS
except NameError:
    from IPython.display import display, HTML
    display(HTML(
        '<div style="border-left:4px solid #e53935;padding:10px;background:#fff5f5;">'
        '<strong style="color:#e53935;">Data loading failed.</strong>'
        ' Check that the zip file is accessible.</div>'
    ))
    raise


In [ ]:
%%capture _evidence_log
# @title D. EvidenceStore, preprocessing, and plot helpers 

# --- EvidenceStore abstraction ---

@dataclass
class EvidencePointer:
    """Traces a result to its source file and filters."""
    file_path: str
    filters: Dict[str, Any] = field(default_factory=dict)
    aggregation: str = ""

    def __str__(self) -> str:
        parts = [f"source: {self.file_path}"]
        if self.filters:
            parts.append("filters: " + ", ".join(f"{k}={v}" for k, v in self.filters.items()))
        if self.aggregation:
            parts.append(f"agg: {self.aggregation}")
        return " | ".join(parts)


@dataclass
class EvidenceResult:
    """Bundles returned data with its provenance."""
    data: Any
    pointer: EvidencePointer


class EvidenceStore:
    """Lazy-loading, cached access to all SYMPHONY bundle artefacts."""

    def __init__(self, zf: zipfile.ZipFile, root_prefix: str, fold_ids: List[str]):
        self._zf = zf
        self._root = root_prefix
        self._folds = fold_ids
        self._cache: Dict[str, Any] = {}
        self._patient_fold_map: Dict[int, str] = {}
        self._build_patient_fold_map()

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------
    def _norm_fold(self, fold: Union[int, str]) -> str:
        if isinstance(fold, int):
            return f"fold_{fold:02d}"
        s = str(fold).strip()
        if re.match(r"^fold_\d+$", s):
            return s
        if s.isdigit():
            return f"fold_{int(s):02d}"
        return s

    def _zpath(self, rel: str) -> str:
        return self._root + rel

    def _read_csv(self, rel: str) -> pd.DataFrame:
        if rel not in self._cache:
            try:
                self._cache[rel] = pd.read_csv(BytesIO(self._zf.read(self._zpath(rel))))
            except KeyError:
                raise FileNotFoundError(f"Not in zip: {rel}")
        return self._cache[rel]

    def _read_json(self, rel: str) -> Any:
        if rel not in self._cache:
            try:
                self._cache[rel] = json.loads(self._zf.read(self._zpath(rel)))
            except KeyError:
                raise FileNotFoundError(f"Not in zip: {rel}")
        return self._cache[rel]

    def _ptr(self, rel: str, filters: Optional[Dict] = None,
             agg: str = "") -> EvidencePointer:
        return EvidencePointer(file_path=rel, filters=filters or {}, aggregation=agg)

    def _build_patient_fold_map(self) -> None:
        try:
            fa = self._read_csv("01_splits/fold_assignments.csv")
            for _, row in fa.iterrows():
                self._patient_fold_map[int(row["patient_id"])] = \
                    f"fold_{int(row['fold']):02d}"
        except Exception:
            pass

    # ------------------------------------------------------------------
    # Fold / patient lookup
    # ------------------------------------------------------------------
    @property
    def folds(self) -> List[str]:
        return list(self._folds)

    def resolve_fold(self, patient_id: int) -> Optional[str]:
        """Look up which test fold a patient belongs to."""
        return self._patient_fold_map.get(int(patient_id))

    def patient_ids_for_fold(self, fold: Union[int, str]) -> List[int]:
        fid = self._norm_fold(fold)
        df = self._read_csv(f"06_translational/{fid}/indices_ui.csv")
        return sorted(df["patient_id"].astype(int).tolist())

    def all_patient_ids(self) -> List[int]:
        """All patient ids across all folds (from fold_assignments)."""
        try:
            fa = self._read_csv("01_splits/fold_assignments.csv")
            return sorted(fa["patient_id"].astype(int).tolist())
        except Exception:
            ids: List[int] = []
            for fid in self._folds:
                ids.extend(self.patient_ids_for_fold(fid))
            return sorted(set(ids))

    # ------------------------------------------------------------------
    # Shared (non-fold) data
    # ------------------------------------------------------------------
    def get_fold_assignments(self) -> EvidenceResult:
        rel = "01_splits/fold_assignments.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_visit_summaries(self) -> EvidenceResult:
        rel = "02_preprocessing/visit_summaries.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_endpoint_completeness(self) -> EvidenceResult:
        rel = "02_preprocessing/endpoint_completeness.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_boundary_prevalence(self) -> EvidenceResult:
        rel = "02_preprocessing/boundary_prevalence.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_omics_selection_report(self) -> EvidenceResult:
        rel = "02_preprocessing/omics_selection_report.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_metadata(self) -> EvidenceResult:
        rel = "metadata.json"
        return EvidenceResult(self._read_json(rel), self._ptr(rel))

    def get_config(self) -> EvidenceResult:
        rel = "config_snapshot.yaml"
        if rel not in self._cache:
            raw = self._zf.read(self._zpath(rel))
            self._cache[rel] = yaml.safe_load(raw)
        return EvidenceResult(self._cache[rel], self._ptr(rel))

    # ------------------------------------------------------------------
    # 02_preprocessing (per-fold)
    # ------------------------------------------------------------------
    def get_standardisation_stats(self, fold: Union[int, str]) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"02_preprocessing/{fid}/standardisation_stats_train.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    # ------------------------------------------------------------------
    # 03_inference
    # ------------------------------------------------------------------
    def get_convergence(self, fold: Union[int, str]) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"03_inference/{fid}/convergence_diagnostics.json"
        return EvidenceResult(self._read_json(rel), self._ptr(rel))

    def get_global_parameters(self, fold: Union[int, str]) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"03_inference/{fid}/global_parameter_summary.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_contraction_ratios(self, fold: Union[int, str]) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"03_inference/{fid}/posterior_contraction_ratios.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_pathway_stability(self, fold: Union[int, str]) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"03_inference/{fid}/pathway_stability.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_L_cond_selected(self, fold: Union[int, str]) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"03_inference/{fid}/L_cond_selected.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_tau_conf_selected(self, fold: Union[int, str]) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"03_inference/{fid}/tau_conf_selected.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    # ------------------------------------------------------------------
    # 04_forecasting
    # ------------------------------------------------------------------
    def get_forecasts(self, fold: Union[int, str],
                      patient_id: Optional[int] = None) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"04_forecasting/{fid}/forecasts_test.csv"
        df = self._read_csv(rel)
        filt: Dict[str, Any] = {}
        if patient_id is not None:
            df = df[df["patient_id"] == int(patient_id)]
            filt["patient_id"] = patient_id
        return EvidenceResult(df, self._ptr(rel, filt))

    def get_conditioning_diagnostics(self, fold: Union[int, str],
                                      patient_id: Optional[int] = None) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"04_forecasting/{fid}/conditioning_diagnostics_test.csv"
        df = self._read_csv(rel)
        filt: Dict[str, Any] = {}
        if patient_id is not None:
            df = df[df["patient_id"] == int(patient_id)]
            filt["patient_id"] = patient_id
        return EvidenceResult(df, self._ptr(rel, filt))

    def get_patient_threshold_event_probs(
        self, fold: Union[int, str], patient_id: int,
        anchor_index: Optional[int] = None,
    ) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"04_forecasting/{fid}/threshold_event_probabilities_test.csv"
        df = self._read_csv(rel)
        filt: Dict[str, Any] = {"patient_id": int(patient_id)}
        df = df[df["patient_id"] == int(patient_id)]
        if anchor_index is not None:
            df = df[df["anchor_index"] == int(anchor_index)]
            filt["anchor_index"] = anchor_index
        return EvidenceResult(df, self._ptr(rel, filt))

    # ------------------------------------------------------------------
    # 05_evaluation
    # ------------------------------------------------------------------
    def get_aggregate_metrics(self, fold: Union[int, str]) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"05_evaluation/{fid}/aggregate_metrics.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_participant_metrics(self, fold: Union[int, str],
                                 patient_id: Optional[int] = None) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"05_evaluation/{fid}/participant_level_metrics.csv"
        df = self._read_csv(rel)
        filt: Dict[str, Any] = {}
        if patient_id is not None:
            df = df[df["patient_id"] == int(patient_id)]
            filt["patient_id"] = patient_id
        return EvidenceResult(df, self._ptr(rel, filt))

    def get_threshold_calibration(self, fold: Union[int, str]) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"05_evaluation/{fid}/threshold_event_calibration.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_reliability_bins(self, fold: Union[int, str]) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"05_evaluation/{fid}/threshold_event_reliability_bins.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_ppc_summary(self, fold: Union[int, str]) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"05_evaluation/{fid}/ppc_summary.json"
        return EvidenceResult(self._read_json(rel), self._ptr(rel))

    def get_suppression_summary(self, fold: Union[int, str]) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"05_evaluation/{fid}/suppression_summary_test.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    # ------------------------------------------------------------------
    # 06_translational
    # ------------------------------------------------------------------
    def get_indices_df(self, fold: Union[int, str]) -> EvidenceResult:
        """Full indices_ui DataFrame for the fold."""
        fid = self._norm_fold(fold)
        rel = f"06_translational/{fid}/indices_ui.csv"
        return EvidenceResult(self._read_csv(rel), self._ptr(rel))

    def get_patient_indices(self, fold: Union[int, str],
                             patient_id: int) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"06_translational/{fid}/indices_ui.csv"
        df = self._read_csv(rel)
        row = df[df["patient_id"] == int(patient_id)]
        if row.empty:
            return EvidenceResult(None, self._ptr(rel, {"patient_id": patient_id},
                                                   "no rows matched"))
        return EvidenceResult(row.iloc[0].to_dict(),
                              self._ptr(rel, {"patient_id": patient_id}))

    def get_patient_latents(self, fold: Union[int, str],
                             patient_id: int) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"06_translational/{fid}/latent_state_trajectories_test.csv"
        df = self._read_csv(rel)
        sub = df[df["patient_id"] == int(patient_id)].copy()
        return EvidenceResult(sub, self._ptr(rel, {"patient_id": patient_id}))

    def get_patient_risk_curves(self, fold: Union[int, str],
                                 patient_id: int) -> EvidenceResult:
        fid = self._norm_fold(fold)
        rel = f"06_translational/{fid}/risk_curves_ui.csv"
        df = self._read_csv(rel)
        sub = df[df["patient_id"] == int(patient_id)].copy()
        return EvidenceResult(sub, self._ptr(rel, {"patient_id": patient_id}))

    # ------------------------------------------------------------------
    # Distribution helpers (for reasoner comparisons)
    # ------------------------------------------------------------------
    def indices_percentile(self, fold: Union[int, str],
                           column: str, value: float) -> Optional[float]:
        """Percentile rank of value within the fold's indices_ui column."""
        fid = self._norm_fold(fold)
        df = self._read_csv(f"06_translational/{fid}/indices_ui.csv")
        if column not in df.columns:
            return None
        col = df[column].dropna()
        if col.empty or pd.isna(value):
            return None
        return float((col < value).sum() / len(col) * 100)

    def indices_distribution(self, fold: Union[int, str],
                              column: str) -> Optional[Dict[str, float]]:
        """Descriptive stats for a column in indices_ui."""
        fid = self._norm_fold(fold)
        df = self._read_csv(f"06_translational/{fid}/indices_ui.csv")
        if column not in df.columns:
            return None
        col = df[column].dropna()
        if col.empty:
            return {"n": 0, "n_missing": int(df[column].isna().sum())}
        return {
            "n": int(len(col)),
            "n_missing": int(df[column].isna().sum()),
            "median": float(col.median()),
            "q25": float(col.quantile(0.25)),
            "q75": float(col.quantile(0.75)),
            "min": float(col.min()),
            "max": float(col.max()),
            "mean": float(col.mean()),
        }

    # ------------------------------------------------------------------
    # Cross-fold aggregation helpers
    # ------------------------------------------------------------------
    def aggregate_metrics_across_folds(self) -> EvidenceResult:
        """Stack aggregate_metrics from all folds with a fold column."""
        frames = []
        for fid in self._folds:
            rel = f"05_evaluation/{fid}/aggregate_metrics.csv"
            df = self._read_csv(rel).copy()
            df.insert(0, "fold", fid)
            frames.append(df)
        combined = pd.concat(frames, ignore_index=True)
        return EvidenceResult(
            combined,
            self._ptr("05_evaluation/*/aggregate_metrics.csv",
                       agg="stacked across all folds"),
        )

    def convergence_across_folds(self) -> EvidenceResult:
        """Key convergence flags from all folds as a DataFrame."""
        rows = []
        for fid in self._folds:
            d = self._read_json(f"03_inference/{fid}/convergence_diagnostics.json")
            gs = d.get("global_summary", {})
            rows.append({
                "fold": fid,
                "max_r_hat": gs.get("max_r_hat"),
                "min_ess_bulk": gs.get("min_ess_bulk"),
                "min_ess_tail": gs.get("min_ess_tail"),
                "divergences_post_warmup": gs.get("total_post_warmup_divergences"),
                "convergence_met": gs.get("convergence_met"),
            })
        return EvidenceResult(
            pd.DataFrame(rows),
            self._ptr("03_inference/*/convergence_diagnostics.json",
                       agg="global_summary across folds"),
        )




# --- Instantiate EvidenceStore and validate ---

STORE = EvidenceStore(zf=ZF, root_prefix=ROOT_PREFIX, fold_ids=FOLD_IDS)

# Compact validation report
_meta = STORE.get_metadata().data

# Per-fold patient counts and file checks
_val_rows = []
for fid in STORE.folds:
    pids = STORE.patient_ids_for_fold(fid)

    # Check key files load without error
    _checks = {}
    for label, fn in [
        ("indices", lambda: STORE.get_indices_df(fid)),
        ("latents", lambda: STORE.get_patient_latents(fid, pids[0]) if pids else None),
        ("agg_metrics", lambda: STORE.get_aggregate_metrics(fid)),
        ("convergence", lambda: STORE.get_convergence(fid)),
        ("ppc", lambda: STORE.get_ppc_summary(fid)),
    ]:
        try:
            r = fn()
            _checks[label] = "ok" if r is not None else "empty"
        except Exception as e:
            _checks[label] = f"ERR: {e}"

    _val_rows.append({
        "fold": fid,
        "n_patients": len(pids),
        **_checks,
    })

_val_df = pd.DataFrame(_val_rows)
_val_styled = _val_df.style.set_caption("Per-fold validation").hide(axis="index")

# Quick convergence check
_conv = STORE.convergence_across_folds().data
_any_fail = (~_conv["convergence_met"].astype(bool)).any()
_convergence_ok = not _any_fail


# Pre-build indices DataFrames for all folds (used by diagnostics)
all_indices_dfs = {}
for _fid in STORE.folds:
    try:
        _idx_res = STORE.get_indices_df(_fid)
        if _idx_res.data is not None and not _idx_res.data.empty:
            all_indices_dfs[_fid] = _idx_res.data
    except Exception:
        pass

# --- Plot helpers ---

# Color scheme: consistent across the notebook
DOMAIN_COLORS = {
    "motor": "#2166ac",
    "cognitive": "#b2182b",
    "nonmotor": "#4dac26",
    "xM": "#2166ac",
    "xC": "#b2182b",
    "xN": "#4dac26",
}
EVENT_COLORS = {
    "dUPDRS>=5": "#6baed6",
    "dUPDRS>=10": "#2166ac",
    "dMoCA<=-2": "#fc8d59",
    "dMoCA<=-3": "#b2182b",
}


def plot_latent_trajectory(
    latents_df: pd.DataFrame,
    patient_id: int,
    ax: Optional[plt.Axes] = None,
) -> Optional[plt.Figure]:
    """
    Plot xM/xC/xN mean over time with IQR bands.
    latents_df: from STORE.get_patient_latents().data
    Returns the Figure if ax was not provided, else None.
    """
    if latents_df is None or latents_df.empty:
        return None

    created_fig = ax is None
    if created_fig:
        fig, ax = plt.subplots(figsize=(8, 3.5))
    else:
        fig = ax.get_figure()

    t = latents_df["t_days_since_baseline"].values
    t_years = t / 365.25

    for key, label in [("xM", "Motor"), ("xC", "Cognitive"), ("xN", "Nonmotor")]:
        mean_col = f"{key}_mean"
        q25_col = f"{key}_q25"
        q75_col = f"{key}_q75"

        if mean_col not in latents_df.columns:
            continue

        color = DOMAIN_COLORS[key]
        ax.plot(t_years, latents_df[mean_col], color=color, linewidth=1.5,
                label=label, marker=".", markersize=4)

        if q25_col in latents_df.columns and q75_col in latents_df.columns:
            ax.fill_between(
                t_years,
                latents_df[q25_col].values,
                latents_df[q75_col].values,
                color=color, alpha=0.15,
            )

    ax.set_xlabel("Years since baseline")
    ax.set_ylabel("Latent state (0-1)")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"Latent trajectory — patient {patient_id}")
    ax.legend(loc="upper left", frameon=True, framealpha=0.9)
    ax.grid(True, alpha=0.3, linewidth=0.5)

    if created_fig:
        fig.tight_layout()
        return fig
    return None


def plot_risk_curves(
    risk_df: pd.DataFrame,
    patient_id: int,
    selected_event: Optional[str] = None,
    ax: Optional[plt.Axes] = None,
) -> Optional[plt.Figure]:
    """
    Plot risk curves: probability vs horizon for each event.
    risk_df: from STORE.get_patient_risk_curves().data
    If selected_event is set, highlight that curve.
    Returns the Figure if ax was not provided, else None.
    """
    if risk_df is None or risk_df.empty:
        return None

    created_fig = ax is None
    if created_fig:
        fig, ax = plt.subplots(figsize=(8, 3.5))
    else:
        fig = ax.get_figure()

    events = risk_df["event"].unique()
    for ev in sorted(events):
        sub = risk_df[risk_df["event"] == ev].sort_values("horizon_days")
        color = EVENT_COLORS.get(ev, "#888888")
        lw = 2.0 if ev == selected_event else 1.2
        alpha = 1.0 if selected_event is None or ev == selected_event else 0.35

        ax.plot(sub["horizon_days"], sub["prob"],
                color=color, linewidth=lw, alpha=alpha,
                marker="o", markersize=4, label=ev)

    ax.set_xlabel("Horizon (days)")
    ax.set_ylabel("Probability")
    ax.set_ylim(-0.02, min(1.02, risk_df["prob"].max() * 1.5 + 0.05))
    ax.set_title(f"Risk curves — patient {patient_id}")
    ax.legend(loc="upper left", frameon=True, framealpha=0.9)
    ax.grid(True, alpha=0.3, linewidth=0.5)

    if created_fig:
        fig.tight_layout()
        return fig
    return None


def plot_calibration_bins(
    bins_df: pd.DataFrame,
    fold_id: str,
    event_filter: Optional[str] = None,
    task_filter: Optional[str] = None,
    ax: Optional[plt.Axes] = None,
) -> Optional[plt.Figure]:
    """
    Reliability diagram from threshold_event_reliability_bins.csv.
    Expected columns: bin_mid (or predicted_mean), observed_freq, n_in_bin.
    Filters to a single event/task if specified.
    Returns the Figure if ax was not provided, else None.
    """
    if bins_df is None or bins_df.empty:
        return None

    df = bins_df.copy()
    if event_filter and "event" in df.columns:
        df = df[df["event"] == event_filter]
    if task_filter and "task" in df.columns:
        df = df[df["task"] == task_filter]
    if df.empty:
        return None

    # Resolve column names (bundles may vary)
    pred_col = None
    for c in ["bin_mid", "predicted_mean", "bin_center", "pred_mean"]:
        if c in df.columns:
            pred_col = c
            break
    obs_col = None
    for c in ["observed_freq", "observed_rate", "obs_freq", "empirical_freq"]:
        if c in df.columns:
            obs_col = c
            break
    n_col = None
    for c in ["n_in_bin", "n", "count"]:
        if c in df.columns:
            n_col = c
            break

    if pred_col is None or obs_col is None:
        return None

    created_fig = ax is None
    if created_fig:
        fig, ax = plt.subplots(figsize=(4.5, 4.5))
    else:
        fig = ax.get_figure()

    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5, label="Perfect")
    sizes = df[n_col].values * 3 if n_col else 30
    ax.scatter(df[pred_col], df[obs_col], s=sizes, color="#2166ac",
               alpha=0.7, edgecolors="white", linewidths=0.5)
    ax.plot(df[pred_col].values, df[obs_col].values, color="#2166ac",
            linewidth=1.0, alpha=0.6)

    title_parts = [f"Reliability — {fold_id}"]
    if event_filter:
        title_parts.append(event_filter)
    if task_filter:
        title_parts.append(task_filter)
    ax.set_title(" / ".join(title_parts), fontsize=10)
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Observed frequency")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_aspect("equal")
    ax.legend(loc="lower right", fontsize=7, frameon=True)
    ax.grid(True, alpha=0.3, linewidth=0.5)

    if created_fig:
        fig.tight_layout()
        return fig
    return None


def plot_aggregate_metrics_bar(
    agg_df: pd.DataFrame,
    metric: str = "MAE",
    endpoint_filter: Optional[str] = None,
    ax: Optional[plt.Axes] = None,
) -> Optional[plt.Figure]:
    """
    Grouped bar chart of a metric across tasks from aggregate_metrics.csv.
    One group per task, colored by endpoint.
    """
    if agg_df is None or agg_df.empty or metric not in agg_df.columns:
        return None

    df = agg_df.copy()
    if endpoint_filter:
        ep_map = {"updrs3": "updrs3", "moca": "moca", "scopa_aut": "scopa_aut",
                   "U": "updrs3", "Q": "moca", "S": "scopa_aut"}
        target = ep_map.get(endpoint_filter, endpoint_filter)
        df = df[df["endpoint"].str.lower().str.contains(target.lower())]
    if df.empty:
        return None

    # Filter to reliable rows if column exists
    if "reliable" in df.columns:
        df = df[df["reliable"].astype(str).str.lower().isin(["true", "1"])]
        if df.empty:
            return None

    created_fig = ax is None
    if created_fig:
        fig, ax = plt.subplots(figsize=(8, 3.5))
    else:
        fig = ax.get_figure()

    ep_colors = {"updrs3": DOMAIN_COLORS["motor"],
                 "moca": DOMAIN_COLORS["cognitive"],
                 "scopa_aut": DOMAIN_COLORS["nonmotor"]}

    tasks = df["task"].unique()
    endpoints = df["endpoint"].unique()
    x = np.arange(len(tasks))
    width = 0.8 / max(len(endpoints), 1)

    for i, ep in enumerate(endpoints):
        sub = df[df["endpoint"] == ep]
        vals = []
        for t in tasks:
            row = sub[sub["task"] == t]
            vals.append(row[metric].values[0] if not row.empty else 0)
        color = ep_colors.get(ep.lower(), "#888888")
        ax.bar(x + i * width, vals, width, label=ep, color=color, alpha=0.8)

    ax.set_xticks(x + width * (len(endpoints) - 1) / 2)
    ax.set_xticklabels(tasks, rotation=15, ha="right")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} by task and endpoint")
    ax.legend(frameon=True, framealpha=0.9)
    ax.grid(True, axis="y", alpha=0.3, linewidth=0.5)

    if created_fig:
        fig.tight_layout()
        return fig
    return None


print("Plot helpers defined.")


In [ ]:
%%capture _helpers_log
# @title D2. Clinical helpers, explorer, and reasoner 

_FOLD_LABELS = {fid: f"Cohort {i+1} ({fid})" for i, fid in enumerate(STORE.folds)}
_FOLD_REVERSE = {v: k for k, v in _FOLD_LABELS.items()}

_EVENT_LABELS = {
    "dUPDRS>=5": "Motor worsening (UPDRS +5)",
    "dUPDRS>=10": "Motor worsening (UPDRS +10)",
    "dMoCA<=-2": "Cognitive decline (MoCA -2)",
    "dMoCA<=-3": "Cognitive decline (MoCA -3)",
}
_EVENT_REVERSE = {v: k for k, v in _EVENT_LABELS.items()}

_ENDPOINT_LABELS = {
    "updrs3": "Motor (UPDRS-III)",
    "moca": "Cognitive (MoCA)",
    "scopa_aut": "Autonomic (SCOPA-AUT)",
}
_ENDPOINT_REVERSE = {v: k for k, v in _ENDPOINT_LABELS.items()}

_IMPROVEMENT_GUIDANCE = {
    "rule1_fewer_than_two_visits": (
        "This patient currently has only one recorded visit. Collecting at least one "
        "additional clinic visit with motor, cognitive and autonomic assessments "
        "would allow the model to estimate a personalised trajectory and generate forecasts."
    ),
    "rule2_incomplete_anchor": (
        "The most recent eligible visit is missing one or more assessment scores "
        "(UPDRS-III, MoCA or SCOPA-AUT). Completing all three assessments at a future "
        "visit would enable the model to condition on a full clinical snapshot."
    ),
    "rule3_low_confidence": (
        "The model's uncertainty about this patient's current disease state is above the "
        "reliability threshold. Additional clinic visits, especially with complete "
        "assessments closer together in time, would help the model narrow its estimate."
    ),
    "rule4_insufficient_runs": (
        "The internal conditioning procedure did not converge reliably at the last anchor. "
        "This is uncommon and may resolve with additional longitudinal data points. "
        "A follow-up visit with complete assessments would improve stability."
    ),
    "rule5_boundary_anchor": (
        "One or more scores at the last anchor visit are at the measurement floor or ceiling "
        "(for example, UPDRS-III = 0 or MoCA = 30). The model performs best when scores "
        "are away from instrument boundaries. A follow-up visit where scores are in the "
        "mid-range would produce more reliable forecasts."
    ),
    "none": "Forecasts are available and no additional data requirements are indicated.",
    "": "Forecast availability status not recorded for this patient.",
}

_COHORT_LABELS = {
    "Parkinson's Disease": "Parkinson's disease (PD)",
    "Prodromal": "Prodromal (at-risk)",
    "Healthy Control": "Healthy control",
    "SWEDD": "SWEDD",
}

# ============================================================================
# Demographic lookup helpers (cached)
# ============================================================================
_demo_cache: Dict[int, Dict[str, Any]] = {}
_visit_cache: Dict[int, Dict[str, Any]] = {}

def _get_demographics(patient_id: int) -> Dict[str, Any]:
    if patient_id in _demo_cache:
        return _demo_cache[patient_id]
    try:
        fa = STORE.get_fold_assignments().data
        row = fa[fa["patient_id"] == int(patient_id)]
        if row.empty:
            _demo_cache[patient_id] = {}
            return {}
        d = row.iloc[0].to_dict()
        _demo_cache[patient_id] = d
        return d
    except Exception:
        return {}

def _get_visit_summary(patient_id: int) -> Dict[str, Any]:
    if patient_id in _visit_cache:
        return _visit_cache[patient_id]
    try:
        vs = STORE.get_visit_summaries().data
        row = vs[vs["patient_id"] == int(patient_id)]
        if row.empty:
            _visit_cache[patient_id] = {}
            return {}
        d = row.iloc[0].to_dict()
        _visit_cache[patient_id] = d
        return d
    except Exception:
        return {}

def _sex_label(sex: Any) -> str:
    if sex in ("M", "m", 1, "1"):
        return "Male"
    if sex in ("F", "f", 0, "0"):
        return "Female"
    return "Not recorded"

def _sex_code(sex: Any) -> str:
    if sex in ("M", "m", 1, "1"):
        return "M"
    if sex in ("F", "f", 0, "0"):
        return "F"
    return "?"

# Build sex lookup per fold for filtering
_sex_by_fold: Dict[str, Dict[int, str]] = {}
try:
    _fa_all = STORE.get_fold_assignments().data
    for fid in STORE.folds:
        fold_num = int(fid.replace("fold_", ""))
        subset = _fa_all[_fa_all["fold"] == fold_num]
        _sex_by_fold[fid] = {
            int(r["patient_id"]): _sex_code(r["sex"]) for _, r in subset.iterrows()
        }
except Exception:
    pass

# ============================================================================
# Widget definitions
# ============================================================================
_WIDGET_STYLE = {"description_width": "auto"}

_w_fold = widgets.Dropdown(
    options=list(_FOLD_LABELS.values()),
    value=list(_FOLD_LABELS.values())[0],
    description="Validation cohort:",
    layout=widgets.Layout(width="260px"),
    style=_WIDGET_STYLE,
)

_w_sex_filter = widgets.Dropdown(
    options=["All patients", "Female only", "Male only"],
    value="All patients",
    description="Sex:",
    layout=widgets.Layout(width="180px"),
    style=_WIDGET_STYLE,
)

_first_fold = STORE.folds[0]
_first_pids = STORE.patient_ids_for_fold(_first_fold)

_w_patient = widgets.Dropdown(
    options=_first_pids,
    value=_first_pids[0] if _first_pids else None,
    description="Patient ID:",
    layout=widgets.Layout(width="230px"),
    style=_WIDGET_STYLE,
)

_w_event = widgets.Dropdown(
    options=list(_EVENT_LABELS.values()),
    value=_EVENT_LABELS["dUPDRS>=10"],
    description="Clinical event:",
    layout=widgets.Layout(width="280px"),
    style=_WIDGET_STYLE,
)

_w_endpoint = widgets.Dropdown(
    options=list(_ENDPOINT_LABELS.values()),
    value=_ENDPOINT_LABELS["updrs3"],
    description="Endpoint:",
    layout=widgets.Layout(width="230px"),
    style=_WIDGET_STYLE,
)

_w_tab = widgets.ToggleButtons(
    options=["Patient profile", "Disease trajectory", "Risk estimates", "Model reliability"],
    value="Patient profile",
    description="",
    layout=widgets.Layout(width="auto"),
    button_style="",
    style={"button_width": "160px", "font_weight": "normal"},
)

_w_out = widgets.Output()

# ============================================================================
# Filtered patient list logic
# ============================================================================
def _filtered_patients(fold_id: str, sex_filter: str) -> List[int]:
    all_pids = STORE.patient_ids_for_fold(fold_id)
    if sex_filter == "All patients":
        return all_pids
    sex_map = _sex_by_fold.get(fold_id, {})
    target = "F" if sex_filter == "Female only" else "M"
    return [p for p in all_pids if sex_map.get(p, "?") == target]

def _refresh_patient_list() -> None:
    fold_id = _FOLD_REVERSE.get(_w_fold.value, STORE.folds[0])
    pids = _filtered_patients(fold_id, _w_sex_filter.value)
    _w_patient.options = pids
    if pids:
        _w_patient.value = pids[0]

def _on_fold_change(change: Dict) -> None:
    _refresh_patient_list()

def _on_sex_change(change: Dict) -> None:
    _refresh_patient_list()

_w_fold.observe(_on_fold_change, names="value")
_w_sex_filter.observe(_on_sex_change, names="value")

# ============================================================================
# Formatting helpers
# ============================================================================
def _cfmt(v: Any, unit: str = "", precision: int = 3) -> str:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "not available (insufficient data)"
    if isinstance(v, float):
        s = f"{v:.{precision}f}"
    else:
        s = str(v)
    return f"{s} {unit}".strip()

def _flag_text(v: Any, true_label: str, false_label: str,
               na_label: str = "not assessed") -> str:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return na_label
    return true_label if int(v) == 1 else false_label

def _severity_word(val: float, thresholds: Tuple[float, ...] = (0.25, 0.50, 0.75),
                   labels: Tuple[str, ...] = ("mild", "moderate", "substantial", "severe")) -> str:
    for t, l in zip(thresholds, labels):
        if val < t:
            return l
    return labels[-1]

def _percentile_context(store: EvidenceStore, fold: str, col: str, val: Any) -> str:
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return ""
    pct = store.indices_percentile(fold, col, float(val))
    if pct is None:
        return ""
    return f"(percentile {pct:.0f} within this cohort)"

def _explain_missing_index(d: Dict, index_name: str) -> str:
    n_vis = d.get("n_visits")
    supp = d.get("suppressed_at_last_anchor")
    reason = d.get("suppression_reason_last_anchor", "")
    anchor_complete = d.get("anchor_complete_all3_last_anchor")

    causes: List[str] = []
    if isinstance(n_vis, (int, float)) and n_vis < 2:
        causes.append(f"only {int(n_vis)} visit recorded; at least 2 are needed")
    if supp == 1 and reason:
        guidance = _IMPROVEMENT_GUIDANCE.get(str(reason), "")
        if guidance:
            causes.append(guidance.split(".")[0].lower())
    if anchor_complete == 0 and not any("missing" in c for c in causes):
        causes.append("the most recent anchor is missing one or more assessment scores")

    if causes:
        return f"{index_name} not yet computable: {'; '.join(causes)}."
    return f"{index_name} not yet computable (additional clinical data needed)."

def _obs_fraction_bar(observed: Any, total: Any) -> str:
    if not isinstance(observed, (int, float)) or not isinstance(total, (int, float)) or total == 0:
        return "no data"
    pct = observed / total * 100
    return f"{int(observed)}/{int(total)} ({pct:.0f}%)"

def _pronoun(sex: str) -> Tuple[str, str, str]:
    """Returns (subject, object, possessive) based on sex."""
    if sex == "Male":
        return ("he", "him", "his")
    if sex == "Female":
        return ("she", "her", "her")
    return ("they", "them", "their")

# ============================================================================
# Tab renderers
# ============================================================================
def _render_patient_profile(fold: str, pid: int) -> None:
    res = STORE.get_patient_indices(fold, pid)
    if res.data is None:
        print(f"Patient {pid} not found in this validation cohort.")
        return

    d = res.data
    demo = _get_demographics(pid)
    visit_info = _get_visit_summary(pid)

    sex = _sex_label(demo.get("sex"))
    age = demo.get("age_baseline_years")
    cohort_raw = demo.get("cohort", visit_info.get("cohort", ""))
    cohort = _COHORT_LABELS.get(cohort_raw, cohort_raw) if cohort_raw else "not recorded"
    has_omics = demo.get("has_omics")
    subj, obj, poss = _pronoun(sex)

    age_str = f"{age:.1f} years at baseline" if isinstance(age, (int, float)) else "age not recorded"
    omics_str = "genetic/omics data available" if has_omics == 1 else (
        "no genetic/omics data" if has_omics == 0 else "omics status unknown"
    )

    display(HTML(f"""
    <div style="background: #f5f5f5; border-left: 4px solid #0f3460; padding: 12px 16px;
                margin-bottom: 10px; border-radius: 2px;">
        <strong style="font-size: 15px;">Patient {pid}</strong>
        <span style="color: #555; margin-left: 14px; font-size: 12px;">
            {_FOLD_LABELS.get(fold, fold)}
        </span>
        <div style="margin-top: 6px; font-size: 13px; color: #333;">
            {sex} &nbsp;|&nbsp; {age_str} &nbsp;|&nbsp; {cohort} &nbsp;|&nbsp; {omics_str}
        </div>
    </div>
    """))

    # Clinical observation summary
    n_vis = d.get("n_visits", None)
    fu = d.get("follow_up_days", None)
    fu_years = f"({fu/365.25:.1f} years)" if isinstance(fu, (int, float)) else ""

    n_updrs = visit_info.get("n_updrs_observed", None)
    n_moca = visit_info.get("n_moca_observed", None)
    n_scopa = visit_info.get("n_scopa_observed", None)
    n_total = visit_info.get("n_visits", n_vis)

    obs_rows = [
        ("Clinic visits", _cfmt(n_vis), fu_years),
        ("Follow-up duration", _cfmt(fu, "days") if fu else "not available", ""),
        ("Motor assessments (UPDRS-III)", _obs_fraction_bar(n_updrs, n_total),
         "of total visits"),
        ("Cognitive assessments (MoCA)", _obs_fraction_bar(n_moca, n_total),
         "of total visits"),
        ("Autonomic assessments (SCOPA-AUT)", _obs_fraction_bar(n_scopa, n_total),
         "of total visits"),
    ]
    obs_df = pd.DataFrame(obs_rows, columns=["Measure", "Value", "Note"])
    display(HTML("<strong style='font-size:12px; color:#333;'>Clinical observations</strong>"))
    display(obs_df.style.hide(axis="index").set_properties(
        **{"font-size": "12px", "text-align": "left"}
    ).set_table_styles([
        {"selector": "th", "props": [("font-size", "11px"), ("color", "#555")]},
    ]))
    print()

    # Latest disease state
    xm = d.get("xM_last")
    xc = d.get("xC_last")
    xn = d.get("xN_last")

    state_rows = []
    for label, val, col_name, thresholds in [
        ("Motor burden (xM)", xm, "xM_last", (0.2, 0.4, 0.6)),
        ("Cognitive burden (xC)", xc, "xC_last", (0.15, 0.3, 0.5)),
        ("Autonomic burden (xN)", xn, "xN_last", (0.2, 0.4, 0.6)),
    ]:
        if isinstance(val, float) and not np.isnan(val):
            sev = _severity_word(val, thresholds)
            pctx = _percentile_context(STORE, fold, col_name, val)
            state_rows.append((label, f"{val:.3f}", sev, pctx))
        else:
            state_rows.append((label, "not yet estimated", "", ""))

    state_df = pd.DataFrame(state_rows,
                            columns=["Domain", "Latent state (0\u20131)", "Severity", "Cohort context"])
    display(HTML("<strong style='font-size:12px; color:#333;'>Latest disease state</strong>"))
    display(state_df.style.hide(axis="index").set_properties(
        **{"font-size": "12px", "text-align": "left"}
    ).set_table_styles([
        {"selector": "th", "props": [("font-size", "11px"), ("color", "#555")]},
    ]))
    print()

    # Derived clinical indices
    mv = d.get("motor_volatility_index")
    cf = d.get("near_term_cognitive_fragility_index")
    accel = d.get("trajectory_acceleration_flag")

    idx_rows = []
    if mv is not None and not (isinstance(mv, float) and np.isnan(mv)):
        idx_rows.append((
            "Motor volatility",
            f"{mv:.1f} UPDRS points",
            "Spread of predicted motor scores; higher means less predictable",
            _percentile_context(STORE, fold, "motor_volatility_index", mv),
        ))
    else:
        idx_rows.append(("Motor volatility",
                         _explain_missing_index(d, "Motor volatility"), "", ""))

    if cf is not None and not (isinstance(cf, float) and np.isnan(cf)):
        risk_level = "elevated" if cf > 0.20 else ("moderate" if cf > 0.10 else "low")
        idx_rows.append((
            "Cognitive fragility",
            f"{cf:.1%}",
            f"Near-term probability of meaningful cognitive decline ({risk_level} risk)",
            _percentile_context(STORE, fold, "near_term_cognitive_fragility_index", cf),
        ))
    else:
        idx_rows.append(("Cognitive fragility",
                         _explain_missing_index(d, "Cognitive fragility"), "", ""))

    idx_rows.append((
        "Trajectory acceleration",
        _flag_text(accel, "Detected: worsening rate is increasing over time",
                   "Not detected: progression rate appears stable",
                   "Not yet assessed (insufficient follow-up horizons)"),
        "", "",
    ))

    idx_df = pd.DataFrame(idx_rows,
                          columns=["Index", "Value", "Interpretation", "Cohort rank"])
    display(HTML("<strong style='font-size:12px; color:#333;'>Derived clinical indices</strong>"))
    display(idx_df.style.hide(axis="index").set_properties(
        **{"font-size": "12px", "text-align": "left"}
    ).set_table_styles([
        {"selector": "th", "props": [("font-size", "11px"), ("color", "#555")]},
    ]))
    print()

    # Forecast readiness (positive framing instead of "suppression")
    conf = d.get("confidence_flag")
    supp = d.get("suppressed_at_last_anchor")
    reason = d.get("suppression_reason_last_anchor", "")
    reason_str = str(reason) if reason else ""

    is_ready = (conf == 1 and supp == 0)

    if is_ready:
        status_color = "#4caf50"
        status_label = "Forecast ready: personalised predictions are available for this patient"
    else:
        status_color = "#1565c0"
        status_label = "Additional data needed to generate reliable personalised predictions"

    display(HTML(f"""
    <div style="border-left: 4px solid {status_color}; padding: 8px 14px;
                background: #fafafa; margin-bottom: 6px; border-radius: 2px;">
        <strong style="color: {status_color};">Prediction status: {status_label}</strong>
    </div>
    """))

    if is_ready:
        print(f"  The model has sufficient data to generate personalised forecasts for this patient.")
    else:
        guidance = _IMPROVEMENT_GUIDANCE.get(reason_str,
                   _IMPROVEMENT_GUIDANCE.get("", ""))
        if guidance:
            print(f"  {guidance}")
        # Additional constructive detail
        if conf == 0 and supp != 1:
            print(f"  The model's confidence in {poss} current disease state is below the "
                  f"reliability threshold. More frequent visits or complete assessments "
                  f"at the next visit would help.")

    # Gating detail (compact, factual)
    delta_iqr = d.get("delta_latent_iqr_last_anchor")
    tau = d.get("tau_conf_last_anchor")
    passed = d.get("passed_conditional_runs_last_anchor")
    l_cond = d.get("L_cond_last_anchor")
    anchor_complete = d.get("anchor_complete_all3_last_anchor")

    gate_parts = []
    if isinstance(delta_iqr, float) and isinstance(tau, float):
        exceeded = "exceeds" if delta_iqr > tau else "within"
        gate_parts.append(f"latent IQR={delta_iqr:.4f} ({exceeded} threshold {tau:.2f})")
    if passed is not None and l_cond is not None:
        gate_parts.append(f"conditioning: {_cfmt(passed)}/{_cfmt(l_cond)} runs passed")
    if anchor_complete is not None:
        gate_parts.append(
            "anchor complete" if int(anchor_complete) == 1 else "anchor incomplete"
        )
    if gate_parts:
        print(f"  Technical detail: {' | '.join(gate_parts)}")

    evidence_sources = [f"06_translational/{fold}/indices_ui.csv"]
    if demo:
        evidence_sources.append("01_splits/fold_assignments.csv")
    if visit_info:
        evidence_sources.append("02_preprocessing/visit_summaries.csv")

    display(HTML(
        f"<p style='color:#999; font-size:11px; margin-top:8px;'>"
        f"Evidence: {' + '.join(evidence_sources)}, patient_id={pid}</p>"
    ))


def _render_trajectory(fold: str, pid: int) -> None:
    res = STORE.get_patient_latents(fold, pid)
    df = res.data
    demo = _get_demographics(pid)

    if df is None or df.empty:
        print(f"No trajectory data for patient {pid}.")
        idx_res = STORE.get_patient_indices(fold, pid)
        if idx_res.data is not None:
            d = idx_res.data
            if isinstance(d.get("n_visits"), (int, float)) and d["n_visits"] < 2:
                print("  This patient has fewer than 2 visits. At least one additional "
                      "clinic visit would allow the model to estimate a trajectory.")
        return

    sex = _sex_label(demo.get("sex"))
    age = demo.get("age_baseline_years")
    cohort_raw = demo.get("cohort", "")
    age_str = f"age {age:.0f}" if isinstance(age, (int, float)) else ""
    subtitle_parts = [p for p in [sex, age_str, cohort_raw] if p]
    subtitle = " | ".join(subtitle_parts)

    fig, ax = plt.subplots(figsize=(9, 4))
    t = df["t_days_since_baseline"].values
    t_years = t / 365.25

    for key, label in [
        ("xM", "Motor (xM)"),
        ("xC", "Cognitive (xC)"),
        ("xN", "Autonomic (xN)"),
    ]:
        mean_col, q25_col, q75_col = f"{key}_mean", f"{key}_q25", f"{key}_q75"
        if mean_col not in df.columns:
            continue
        color = DOMAIN_COLORS[key]
        ax.plot(t_years, df[mean_col], color=color, linewidth=1.8,
                label=label, marker=".", markersize=5, zorder=3)
        if q25_col in df.columns and q75_col in df.columns:
            ax.fill_between(t_years, df[q25_col].values, df[q75_col].values,
                            color=color, alpha=0.12, zorder=1)

    ax.set_xlabel("Years since baseline", fontsize=10)
    ax.set_ylabel("Latent disease burden (0 = none, 1 = maximum)", fontsize=10)
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"Disease progression trajectory: patient {pid}", fontsize=12, fontweight="bold")
    if subtitle:
        ax.text(0.5, 1.02, subtitle, transform=ax.transAxes, ha="center",
                fontsize=9, color="#777", style="italic")
    ax.legend(loc="upper left", frameon=True, framealpha=0.95, fontsize=9)
    ax.grid(True, alpha=0.25, linewidth=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout(rect=[0, 0, 1, 0.97] if subtitle else None)
    plt.show()
    plt.close(fig)

    last = df.iloc[-1]
    xm_last = last.get("xM_mean", np.nan)
    xc_last = last.get("xC_mean", np.nan)
    xn_last = last.get("xN_mean", np.nan)
    fu_yrs = t_years[-1] if len(t_years) > 0 else 0

    vals = {"motor": xm_last, "cognitive": xc_last, "autonomic": xn_last}
    valid_vals = {k: v for k, v in vals.items() if isinstance(v, float) and not np.isnan(v)}
    dominant = max(valid_vals, key=valid_vals.get) if valid_vals else "unknown"

    subj, obj, poss = _pronoun(sex)

    display(HTML(f"""
    <div style="background: #f9f9f9; padding: 8px 14px; border-radius: 4px;
                font-size: 12px; color: #444; margin-top: 4px;">
        <strong>How to read this plot:</strong> each line tracks the model's estimate of latent
        disease burden over time (0 = no impairment, 1 = maximum). Shaded bands show
        posterior uncertainty (interquartile range). Over <strong>{fu_yrs:.1f} years</strong>
        and <strong>{len(df)} visits</strong>, this patient shows the highest burden in the
        <strong>{dominant}</strong> domain (motor={_cfmt(xm_last, precision=2)},
        cognitive={_cfmt(xc_last, precision=2)}, autonomic={_cfmt(xn_last, precision=2)}).
    </div>
    <p style='color:#999; font-size:11px; margin-top:4px;'>Evidence: {res.pointer}</p>
    """))


def _render_risk_estimates(fold: str, pid: int, event_label: str) -> None:
    event = _EVENT_REVERSE.get(event_label, event_label)
    res = STORE.get_patient_risk_curves(fold, pid)
    df = res.data
    demo = _get_demographics(pid)
    sex = _sex_label(demo.get("sex"))
    subj, obj, poss = _pronoun(sex)

    if df is None or df.empty:
        print(f"No risk estimates available for patient {pid}.")
        idx_res = STORE.get_patient_indices(fold, pid)
        if idx_res.data is not None:
            d = idx_res.data
            n_vis = d.get("n_visits", None)
            reason = d.get("suppression_reason_last_anchor", "")
            reason_str = str(reason) if reason else ""

            print()
            print("  What would be needed to generate risk estimates:")
            guidance = _IMPROVEMENT_GUIDANCE.get(reason_str, "")
            if guidance:
                print(f"  {guidance}")
            elif isinstance(n_vis, (int, float)) and n_vis < 2:
                print(f"  {_IMPROVEMENT_GUIDANCE['rule1_fewer_than_two_visits']}")
            else:
                print("  Additional clinic visits with complete assessments would enable "
                      "the model to produce personalised risk estimates.")
        return

    # Patient context
    age = demo.get("age_baseline_years")
    age_str = f"age {age:.0f}" if isinstance(age, (int, float)) else ""
    context_parts = [p for p in [sex, age_str, demo.get("cohort", "")] if p]

    # Pivot table with clinical labels
    pivot = df.pivot_table(index="event", columns="horizon_days",
                           values="prob", aggfunc="first")
    new_cols = {}
    for h in pivot.columns:
        if h <= 90:
            new_cols[h] = f"{h}d (~{h/30:.0f}mo)"
        else:
            new_cols[h] = f"{h}d (~{h/365.25:.1f}yr)"
    pivot = pivot.rename(columns=new_cols)
    pivot.index = [_EVENT_LABELS.get(e, e) for e in pivot.index]

    ctx_str = f" ({', '.join(context_parts)})" if context_parts else ""
    print(f"Estimated event probabilities for patient {pid}{ctx_str}")
    display(
        pivot.style
        .format(lambda x: f"{x:.1%}" if pd.notna(x) else "\u2014")
        .set_properties(**{"font-size": "12px", "text-align": "center"})
        .background_gradient(cmap="YlOrRd", vmin=0, vmax=0.5, axis=None)
        .set_table_styles([
            {"selector": "th", "props": [("font-size", "11px")]},
        ])
    )
    print()

    # Plot
    fig, ax = plt.subplots(figsize=(9, 4))
    events_in_data = df["event"].unique()
    for ev in sorted(events_in_data):
        sub = df[df["event"] == ev].sort_values("horizon_days")
        color = EVENT_COLORS.get(ev, "#888888")
        is_selected = (ev == event)
        lw = 2.2 if is_selected else 1.0
        alpha = 1.0 if is_selected else 0.3
        ms = 7 if is_selected else 4
        zorder = 5 if is_selected else 2
        label = _EVENT_LABELS.get(ev, ev)
        ax.plot(sub["horizon_days"], sub["prob"], color=color, linewidth=lw,
                alpha=alpha, marker="o", markersize=ms, label=label, zorder=zorder)

    ax.set_xlabel("Forecast horizon (days)", fontsize=10)
    ax.set_ylabel("Predicted probability", fontsize=10)
    ax.set_ylim(-0.02, min(1.02, df["prob"].max() * 1.4 + 0.05))
    ax.set_title(f"Risk curves: patient {pid}", fontsize=12, fontweight="bold")
    if context_parts:
        ax.text(0.5, 1.02, ", ".join(context_parts), transform=ax.transAxes,
                ha="center", fontsize=9, color="#777", style="italic")
    ax.legend(loc="upper left", frameon=True, framealpha=0.95, fontsize=8)
    ax.grid(True, alpha=0.25, linewidth=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout(rect=[0, 0, 1, 0.97] if context_parts else None)
    plt.show()
    plt.close(fig)

    sel_data = df[df["event"] == event].sort_values("horizon_days")
    if not sel_data.empty:
        last_h = sel_data.iloc[-1]
        first_h = sel_data.iloc[0]
        display(HTML(f"""
        <div style="background: #f9f9f9; padding: 8px 14px; border-radius: 4px;
                    font-size: 12px; color: #444; margin-top: 4px;">
            <strong>How to read this plot:</strong> each curve shows the predicted probability
            of a clinical event at increasing time horizons. The highlighted curve
            (<em>{_EVENT_LABELS.get(event, event)}</em>) ranges from
            <strong>{first_h['prob']:.1%}</strong> at {int(first_h['horizon_days'])} days to
            <strong>{last_h['prob']:.1%}</strong> at {int(last_h['horizon_days'])} days.
            Probabilities above approximately 25% are generally considered clinically noteworthy.
        </div>
        <p style='color:#999; font-size:11px; margin-top:4px;'>Evidence: {res.pointer}</p>
        """))
    else:
        display(HTML(f"<p style='color:#999; font-size:11px;'>Evidence: {res.pointer}</p>"))


def _render_model_reliability(fold: str, endpoint_label: str) -> None:
    endpoint = _ENDPOINT_REVERSE.get(endpoint_label, endpoint_label)

    # Aggregate metrics
    agg_res = STORE.get_aggregate_metrics(fold)
    agg = agg_res.data
    agg_filt = agg[agg["endpoint"].str.lower().str.contains(endpoint.lower())]

    if "reliable" in agg_filt.columns:
        agg_filt = agg_filt.copy()
        agg_filt["reliable"] = agg_filt["reliable"].map(
            {True: "Yes", False: "No", "True": "Yes", "False": "No"}
        )

    if not agg_filt.empty:
        show_cols = [c for c in ["task", "endpoint", "MAE", "RMSE", "CRPS",
                                  "coverage95", "coverage90",
                                  "mean_interval_width_95", "n_instances", "reliable"]
                     if c in agg_filt.columns]
        print(f"Forecast accuracy: {_FOLD_LABELS.get(fold, fold)}, {endpoint_label}")
        display(
            agg_filt[show_cols].reset_index(drop=True).style
            .format({
                "MAE": "{:.2f}", "RMSE": "{:.2f}", "CRPS": "{:.2f}",
                "coverage95": "{:.1%}", "coverage90": "{:.1%}",
                "mean_interval_width_95": "{:.1f}",
            }, na_rep="\u2014")
            .set_properties(**{"font-size": "12px", "text-align": "center"})
            .set_table_styles([
                {"selector": "th", "props": [("font-size", "11px"), ("color", "#555")]},
            ])
        )
        display(HTML(f"<p style='color:#999; font-size:11px;'>Evidence: {agg_res.pointer}</p>"))
    else:
        print(f"No metrics found for {endpoint_label} in this cohort.")
    print()

    # Calibration summary
    try:
        cal_res = STORE.get_threshold_calibration(fold)
        cal = cal_res.data
        if not cal.empty:
            cal_display = cal.copy()
            rename_map = {
                "event": "Clinical event", "task": "Forecast task",
                "brier_score": "Brier score",
                "brier_skill_score": "Brier skill (vs base rate)",
                "expected_calibration_error": "Calibration error (ECE)",
                "n_instances": "N",
            }
            cal_display = cal_display.rename(
                columns={k: v for k, v in rename_map.items() if k in cal_display.columns}
            )
            if "Clinical event" in cal_display.columns:
                cal_display["Clinical event"] = cal_display["Clinical event"].map(
                    lambda x: _EVENT_LABELS.get(x, x)
                )
            print(f"Threshold event calibration: {_FOLD_LABELS.get(fold, fold)}")
            display(
                cal_display.reset_index(drop=True).style
                .format(precision=4, na_rep="\u2014")
                .set_properties(**{"font-size": "12px", "text-align": "center"})
                .set_table_styles([
                    {"selector": "th", "props": [("font-size", "11px"), ("color", "#555")]},
                ])
            )
            display(HTML(
                f"<p style='color:#999; font-size:11px;'>Evidence: {cal_res.pointer}</p>"
            ))
        print()
    except Exception:
        pass

    # Calibration bins plot
    try:
        bins_res = STORE.get_reliability_bins(fold)
        bins_df = bins_res.data
        if bins_df is not None and not bins_df.empty:
            ev_for_plot = bins_df["event"].iloc[0] if "event" in bins_df.columns else None
            task_for_plot = bins_df["task"].iloc[0] if "task" in bins_df.columns else None
            fig = plot_calibration_bins(bins_df, fold, ev_for_plot, task_for_plot)
            if fig is not None:
                plt.show()
                plt.close(fig)
                display(HTML(f"""
                <div style="background: #f9f9f9; padding: 8px 14px; border-radius: 4px;
                            font-size: 12px; color: #444; margin-top: 4px;">
                    <strong>How to read this plot:</strong> points on the diagonal mean predicted
                    probabilities match observed event rates (perfect calibration). Points above
                    the diagonal indicate underconfidence; below indicates overconfidence.
                    Dot size reflects the number of patients in each bin.
                </div>
                <p style='color:#999; font-size:11px; margin-top:4px;'>
                    Evidence: {bins_res.pointer}</p>
                """))
        print()
    except Exception:
        pass

    # Convergence summary
    try:
        conv_res = STORE.get_convergence(fold)
        gs = conv_res.data.get("global_summary", {})
        met = gs.get("convergence_met", None)
        status_color = "#4caf50" if met else "#e53935"
        status_label = "All criteria met" if met else "Criteria not met"

        display(HTML(f"""
        <div style="border-left: 4px solid {status_color}; padding: 8px 14px;
                    background: #fafafa; margin: 8px 0; border-radius: 2px;">
            <strong>MCMC convergence for {_FOLD_LABELS.get(fold, fold)}:</strong>
            <span style="color: {status_color};">{status_label}</span>
        </div>
        """))

        conv_rows = [
            ("Max R-hat (target < 1.01)", _cfmt(gs.get("max_r_hat"), precision=4)),
            ("Min effective sample size (bulk)", _cfmt(gs.get("min_ess_bulk"))),
            ("Min effective sample size (tail)", _cfmt(gs.get("min_ess_tail"))),
            ("Post-warmup divergent transitions", _cfmt(gs.get("total_post_warmup_divergences"))),
        ]
        conv_df = pd.DataFrame(conv_rows, columns=["Diagnostic", "Value"])
        display(conv_df.style.hide(axis="index").set_properties(
            **{"font-size": "12px", "text-align": "left"}
        ))
        display(HTML(
            f"<p style='color:#999; font-size:11px;'>Evidence: {conv_res.pointer}</p>"
        ))
    except Exception:
        print("Convergence diagnostics not available for this fold.")


# ============================================================================
# Main dispatch
# ============================================================================
def _on_update(*_args: Any) -> None:
    _w_out.clear_output(wait=True)
    with _w_out:
        fold = _FOLD_REVERSE.get(_w_fold.value, STORE.folds[0])
        pid = _w_patient.value
        tab = _w_tab.value

        if pid is None:
            print("No patient selected.")
            return

        try:
            if tab == "Patient profile":
                _render_patient_profile(fold, pid)
            elif tab == "Disease trajectory":
                _render_trajectory(fold, pid)
            elif tab == "Risk estimates":
                _render_risk_estimates(fold, pid, _w_event.value)
            elif tab == "Model reliability":
                _render_model_reliability(fold, _w_endpoint.value)
        except Exception as e:
            print(f"Error: {e}")
            import traceback
            traceback.print_exc()

for w in [_w_fold, _w_patient, _w_event, _w_endpoint, _w_tab]:
    w.observe(_on_update, names="value")

# ============================================================================
# Layout
# ============================================================================
_row1 = widgets.HBox([_w_fold, _w_sex_filter, _w_patient])
_row2 = widgets.HBox([_w_event, _w_endpoint])
_ui = widgets.VBox([_row1, _row2, _w_tab, _w_out],
                    layout=widgets.Layout(gap="6px"))

display(_ui)
_on_update()

# --- Clinical reasoner (deterministic, grounded) ---

@dataclass
class ReasonerAnswer:
    """Structured answer from the clinical reasoner."""
    clinical_answer: str
    technical_note: str
    evidence: List[EvidencePointer]
    prediction_status: str  # "ready", "additional_data_needed", "not_applicable"
    prediction_guidance: str  # what would improve predictions

    def display(self) -> None:
        # Clinical answer
        display(HTML(f"""
        <div style="border-left: 4px solid #0f3460; padding: 10px 14px;
                    background: #f5f7fa; margin-bottom: 8px; border-radius: 2px;">
            <strong style="font-size: 13px; color: #0f3460;">Clinical answer</strong><br>
            <span style="font-size: 12px; color: #333; line-height: 1.5;">{self.clinical_answer}</span>
        </div>
        """))

        # Prediction status
        if self.prediction_status == "additional_data_needed":
            display(HTML(f"""
            <div style="border-left: 4px solid #1565c0; padding: 8px 14px;
                        background: #f5f5f5; margin-bottom: 8px; border-radius: 2px;">
                <strong style="font-size: 12px; color: #1565c0;">
                    What would improve predictions for this patient</strong><br>
                <span style="font-size: 12px; color: #555;">{self.prediction_guidance}</span>
            </div>
            """))

        # Technical note
        display(HTML(f"""
        <details style="margin-bottom: 6px;">
            <summary style="font-size: 11px; color: #777; cursor: pointer;">
                Technical detail (click to expand)</summary>
            <div style="font-size: 11px; color: #666; padding: 6px 12px;
                        background: #fafafa; border-radius: 2px; margin-top: 4px;">
                {self.technical_note}
            </div>
        </details>
        """))

        # Evidence pointers
        ev_lines = "<br>".join(str(e) for e in self.evidence)
        display(HTML(
            f"<p style='color:#999; font-size:10px; margin-top:4px;'>Evidence: {ev_lines}</p>"
        ))


class ClinicalReasoner:
    """
    Deterministic clinical reasoning over SYMPHONY bundle outputs.
    Parses free-text questions into intents via regex, computes grounded answers,
    and returns structured results with evidence pointers.
    """

    def __init__(self, store: EvidenceStore):
        self._store = store
        self._intent_rules: List[Tuple[str, re.Pattern, Optional[Dict]]] = self._build_rules()

    # ------------------------------------------------------------------
    # Intent detection rules
    # ------------------------------------------------------------------
    def _build_rules(self) -> List[Tuple[str, re.Pattern, Optional[Dict]]]:
        rules = [
            # Risk / probability queries
            ("risk_updrs10", re.compile(
                r"(risk|prob|chance|likelihood).*(motor|updrs).*(10|ten|\+\s*10)", re.I), None),
            ("risk_updrs5", re.compile(
                r"(risk|prob|chance|likelihood).*(motor|updrs).*(5|five|\+\s*5)", re.I), None),
            ("risk_moca2", re.compile(
                r"(risk|prob|chance|likelihood).*(cog|moca).*(2|two)", re.I), None),
            ("risk_moca3", re.compile(
                r"(risk|prob|chance|likelihood).*(cog|moca).*(3|three)", re.I), None),
            ("risk_general", re.compile(
                r"(risk|prob|chance|likelihood).*(worsen|declin|progress|deteri)", re.I), None),

            # Specific indices
            ("cognitive_fragility", re.compile(
                r"(cognitive|cog)\s*(fragil|vulnerab|risk)", re.I), None),
            ("motor_volatility", re.compile(
                r"(motor|updrs)\s*(volatil|variab|spread|predict)", re.I), None),
            ("trajectory_acceleration", re.compile(
                r"(accel|speed.*up|worsen.*faster|rate.*increas)", re.I), None),

            # Trajectory / disease state
            ("trajectory", re.compile(
                r"(trajector|progression|disease\s*state|latent|burden|how.*progress)", re.I), None),

            # Prediction readiness (replaces "suppression" / "confidence")
            ("prediction_status", re.compile(
                r"(predict|forecast|reliab|confiden|trust|suppres|why.*na|why.*miss"
                r"|why.*not\s*avail|additional\s*data|what.*need|ready)", re.I), None),

            # Calibration / model quality
            ("calibration", re.compile(
                r"(calibrat|brier|ece|reliab.*bin|well.*calibrat)", re.I), None),
            ("model_performance", re.compile(
                r"(model|performance|accuracy|mae|rmse|crps|coverage|how.*good"
                r"|how.*accurat|error|metric)", re.I), None),

            # Convergence / inference health
            ("convergence", re.compile(
                r"(converg|rhat|r-hat|ess|divergen|mcmc|sampl|nuts|inference"
                r"|chain|posterior)", re.I), None),

            # Reviewer / audit
            ("reviewer", re.compile(
                r"(review|audit|accept|correct|valid|trust|ppc"
                r"|posterior.*predict|diagnostics\s*ok)", re.I), None),

            # Horizon-specific queries
            ("horizon_specific", re.compile(
                r"(at|in|within|by)\s*(\d+)\s*(day|month|year|d|m|yr)", re.I), None),

            # Comparison / cohort context
            ("comparison", re.compile(
                r"(compar|percentile|relative|cohort|worse.*than|better.*than"
                r"|how.*rank|where.*stand|typical|average)", re.I), None),

            # Patient summary (catch-all)
            ("patient_summary", re.compile(
                r"(summar|overview|tell\s*me\s*about|who\s*is|profile|overall"
                r"|what.*know|describe)", re.I), None),
        ]
        return rules

    # ------------------------------------------------------------------
    # Horizon parsing helper
    # ------------------------------------------------------------------
    def _parse_horizon(self, question: str) -> Optional[int]:
        """Extract a horizon in days from the question text."""
        # Direct day mention
        m = re.search(r"(\d+)\s*(?:day|d)\b", question, re.I)
        if m:
            return int(m.group(1))
        # Month
        m = re.search(r"(\d+)\s*(?:month|mo)\b", question, re.I)
        if m:
            return int(m.group(1)) * 30
        # Year
        m = re.search(r"(\d+)\s*(?:year|yr)\b", question, re.I)
        if m:
            return int(m.group(1)) * 365
        # Named periods
        if re.search(r"\b(1|one)\s*year\b", question, re.I):
            return 365
        if re.search(r"\b(2|two)\s*year\b", question, re.I):
            return 730
        if re.search(r"\b(6|six)\s*month\b", question, re.I):
            return 180
        return None

    def _closest_horizon(self, available: List[int], target: int) -> int:
        """Find the closest available horizon to the requested one."""
        return min(available, key=lambda h: abs(h - target))

    # ------------------------------------------------------------------
    # Shared patient context builder
    # ------------------------------------------------------------------
    def _patient_context(self, fold: str, pid: int) -> Tuple[
        Optional[Dict], Dict, Dict, str, Tuple[str, str, str]
    ]:
        """Returns (indices_dict, demo_dict, visit_dict, prediction_status, pronouns)."""
        idx_res = self._store.get_patient_indices(fold, pid)
        idx = idx_res.data
        demo = _get_demographics(pid)
        visit = _get_visit_summary(pid)
        sex = _sex_label(demo.get("sex"))
        pronouns = _pronoun(sex)

        status = "ready"
        if idx is not None:
            conf = idx.get("confidence_flag")
            supp = idx.get("suppressed_at_last_anchor")
            if supp == 1 or conf == 0:
                status = "additional_data_needed"
        else:
            status = "additional_data_needed"

        return idx, demo, visit, status, pronouns

    def _build_guidance(self, idx: Optional[Dict]) -> str:
        """Construct improvement guidance from patient data."""
        if idx is None:
            return "No index data found for this patient in this cohort."
        reason = str(idx.get("suppression_reason_last_anchor", ""))
        guidance = _IMPROVEMENT_GUIDANCE.get(reason, "")
        if guidance:
            return guidance
        conf = idx.get("confidence_flag")
        if conf == 0:
            return _IMPROVEMENT_GUIDANCE.get("rule3_low_confidence", "")
        return _IMPROVEMENT_GUIDANCE.get("none", "")

    def _severity_label(self, val: float, thresholds=(0.25, 0.5, 0.75),
                        labels=("mild", "moderate", "substantial", "severe")) -> str:
        for t, l in zip(thresholds, labels):
            if val < t:
                return l
        return labels[-1]

    # ------------------------------------------------------------------
    # Intent handlers
    # ------------------------------------------------------------------
    def _handle_risk_event(self, fold: str, pid: int, event: str,
                           event_label: str, question: str) -> ReasonerAnswer:
        idx, demo, visit, status, (subj, obj, poss) = self._patient_context(fold, pid)
        evidence: List[EvidencePointer] = []

        # Get risk curves
        risk_res = self._store.get_patient_risk_curves(fold, pid)
        evidence.append(risk_res.pointer)
        df = risk_res.data

        if df is None or df.empty or event not in df["event"].values:
            # Also try threshold event probabilities
            if idx is not None:
                last_anchor = idx.get("last_anchor_index")
                if last_anchor is not None:
                    tep_res = self._store.get_patient_threshold_event_probs(
                        fold, pid, anchor_index=int(last_anchor)
                    )
                    evidence.append(tep_res.pointer)
                    tep = tep_res.data
                    if tep is not None and not tep.empty:
                        tep_ev = tep[tep["event"] == event]
                        if not tep_ev.empty:
                            df = tep_ev.rename(columns={"horizon_days": "horizon_days"})

            if df is None or df.empty or (isinstance(df, pd.DataFrame) and event not in df.get("event", pd.Series()).values):
                guidance = self._build_guidance(idx)
                return ReasonerAnswer(
                    clinical_answer=(
                        f"Risk estimates for {event_label.lower()} are not currently "
                        f"available for patient {pid}. The model requires additional clinical "
                        f"data before it can generate personalised predictions."
                    ),
                    technical_note=(
                        f"No rows found for event='{event}' in risk_curves_ui.csv or "
                        f"threshold_event_probabilities_test.csv for this patient."
                    ),
                    evidence=evidence,
                    prediction_status=status,
                    prediction_guidance=guidance,
                )

        ev_data = df[df["event"] == event].sort_values("horizon_days")
        horizons = ev_data["horizon_days"].tolist()
        probs = ev_data["prob"].tolist()

        # Check if a specific horizon was requested
        target_h = self._parse_horizon(question)
        if target_h is not None and horizons:
            closest = self._closest_horizon(horizons, target_h)
            prob_at_h = ev_data[ev_data["horizon_days"] == closest]["prob"].values[0]
            horizon_desc = f"{closest} days (~{closest/365.25:.1f} years)"

            # Cohort comparison for cognitive fragility or general context
            pctx = ""
            if "moca" in event.lower():
                col = "near_term_cognitive_fragility_index"
            else:
                col = "motor_volatility_index"
            if idx is not None:
                val = idx.get(col)
                if val is not None and not (isinstance(val, float) and np.isnan(val)):
                    pct = self._store.indices_percentile(fold, col, float(val))
                    if pct is not None:
                        pctx = f" This places {obj} at percentile {pct:.0f} in the cohort for the related index."

            clinical = (
                f"For patient {pid}, the predicted probability of {event_label.lower()} "
                f"within {horizon_desc} is {prob_at_h:.1%}.{pctx}"
            )
        else:
            # Summarise across all horizons
            parts = [f"{h}d: {p:.1%}" for h, p in zip(horizons, probs)]
            clinical = (
                f"For patient {pid}, the predicted probabilities of {event_label.lower()} "
                f"across available horizons are: {', '.join(parts)}."
            )

        # Risk interpretation
        max_prob = max(probs)
        if max_prob >= 0.25:
            clinical += (
                " At least one horizon shows a probability above 25%, which is "
                "generally considered clinically noteworthy."
            )
        elif max_prob >= 0.10:
            clinical += " Probabilities are in the moderate range (10\u201325%)."
        else:
            clinical += " All probabilities are below 10%, indicating relatively low near-term risk."

        technical = (
            f"Event: {event}. Horizons queried: {horizons}. "
            f"Probabilities from risk_curves_ui.csv (or threshold_event_probabilities_test.csv "
            f"if risk curves not available). "
            f"Values are posterior predictive probabilities conditioned on the patient's "
            f"last eligible anchor visit."
        )

        guidance = self._build_guidance(idx)
        return ReasonerAnswer(clinical, technical, evidence, status, guidance)

    def _handle_cognitive_fragility(self, fold: str, pid: int,
                                     question: str) -> ReasonerAnswer:
        idx, demo, visit, status, (subj, obj, poss) = self._patient_context(fold, pid)
        evidence = [self._store.get_patient_indices(fold, pid).pointer]

        if idx is None:
            return ReasonerAnswer(
                f"No data available for patient {pid} in this cohort.",
                "Patient not found in indices_ui.csv.",
                evidence, "additional_data_needed", "Patient not in this validation cohort."
            )

        cf = idx.get("near_term_cognitive_fragility_index")
        guidance = self._build_guidance(idx)

        if cf is None or (isinstance(cf, float) and np.isnan(cf)):
            explanation = _explain_missing_index(idx, "Cognitive fragility index")
            return ReasonerAnswer(
                clinical_answer=(
                    f"The cognitive fragility index is not yet computable for patient {pid}. "
                    f"{explanation}"
                ),
                technical_note=(
                    "near_term_cognitive_fragility_index is NaN in indices_ui.csv. "
                    f"n_visits={idx.get('n_visits')}, "
                    f"anchor_complete={idx.get('anchor_complete_all3_last_anchor')}, "
                    f"suppression_reason={idx.get('suppression_reason_last_anchor')}."
                ),
                evidence=evidence,
                prediction_status="additional_data_needed",
                prediction_guidance=guidance,
            )

        # Cohort comparison
        dist = self._store.indices_distribution(fold, "near_term_cognitive_fragility_index")
        pct = self._store.indices_percentile(fold, "near_term_cognitive_fragility_index", cf)

        risk_level = "elevated" if cf > 0.20 else ("moderate" if cf > 0.10 else "low")
        cohort_ctx = ""
        if dist and pct is not None:
            cohort_ctx = (
                f" Within this cohort, this places {obj} at percentile {pct:.0f} "
                f"(cohort median: {dist['median']:.1%}, IQR: {dist['q25']:.1%}\u2013{dist['q75']:.1%})."
            )

        clinical = (
            f"Patient {pid} has a near-term cognitive fragility index of {cf:.1%}, "
            f"indicating {risk_level} risk of meaningful cognitive decline "
            f"(MoCA drop of 2 or more points) in the near term.{cohort_ctx}"
        )

        technical = (
            f"near_term_cognitive_fragility_index = {cf:.4f} from indices_ui.csv. "
            f"Defined as the peak P(dMoCA <= -2) across available forecast horizons, "
            f"conditioned at the last eligible anchor."
        )

        return ReasonerAnswer(clinical, technical, evidence, status, guidance)

    def _handle_motor_volatility(self, fold: str, pid: int,
                                  question: str) -> ReasonerAnswer:
        idx, demo, visit, status, (subj, obj, poss) = self._patient_context(fold, pid)
        evidence = [self._store.get_patient_indices(fold, pid).pointer]

        if idx is None:
            return ReasonerAnswer(
                f"No data available for patient {pid} in this cohort.",
                "Patient not found in indices_ui.csv.",
                evidence, "additional_data_needed", ""
            )

        mv = idx.get("motor_volatility_index")
        guidance = self._build_guidance(idx)

        if mv is None or (isinstance(mv, float) and np.isnan(mv)):
            explanation = _explain_missing_index(idx, "Motor volatility index")
            return ReasonerAnswer(
                f"The motor volatility index is not yet computable for patient {pid}. {explanation}",
                f"motor_volatility_index is NaN in indices_ui.csv. "
                f"n_visits={idx.get('n_visits')}, "
                f"suppression_reason={idx.get('suppression_reason_last_anchor')}.",
                evidence, "additional_data_needed", guidance,
            )

        dist = self._store.indices_distribution(fold, "motor_volatility_index")
        pct = self._store.indices_percentile(fold, "motor_volatility_index", mv)

        cohort_ctx = ""
        if dist and pct is not None:
            cohort_ctx = (
                f" Within this cohort, {subj} is at percentile {pct:.0f} "
                f"(cohort median: {dist['median']:.1f}, "
                f"IQR: {dist['q25']:.1f}\u2013{dist['q75']:.1f} UPDRS points)."
            )

        clinical = (
            f"Patient {pid} has a motor volatility index of {mv:.1f} UPDRS points. "
            f"This reflects the spread of the model's predicted motor scores at the next "
            f"visit: higher values indicate greater uncertainty in the motor forecast.{cohort_ctx}"
        )

        technical = (
            f"motor_volatility_index = {mv:.2f} from indices_ui.csv. "
            f"Defined as 1.349 * mean(pred_sd) across valid motor forecasts at the last "
            f"anchor, approximating the IQR of the predictive distribution."
        )

        return ReasonerAnswer(clinical, technical, evidence, status, guidance)

    def _handle_trajectory_acceleration(self, fold: str, pid: int,
                                         question: str) -> ReasonerAnswer:
        idx, demo, visit, status, (subj, obj, poss) = self._patient_context(fold, pid)
        evidence = [self._store.get_patient_indices(fold, pid).pointer]

        if idx is None:
            return ReasonerAnswer(
                f"No data available for patient {pid} in this cohort.",
                "Patient not found in indices_ui.csv.",
                evidence, "additional_data_needed", ""
            )

        accel = idx.get("trajectory_acceleration_flag")
        guidance = self._build_guidance(idx)

        if accel is None or (isinstance(accel, float) and np.isnan(accel)):
            explanation = _explain_missing_index(idx, "Trajectory acceleration flag")
            return ReasonerAnswer(
                f"Trajectory acceleration has not been assessed for patient {pid}. {explanation}",
                f"trajectory_acceleration_flag is NaN in indices_ui.csv.",
                evidence, "additional_data_needed", guidance,
            )

        if int(accel) == 1:
            clinical = (
                f"Trajectory acceleration has been detected for patient {pid}. "
                f"This means the rate of disease worsening appears to be increasing "
                f"over time: longer-horizon forecasts show proportionally greater "
                f"deterioration than shorter ones."
            )
        else:
            clinical = (
                f"No trajectory acceleration detected for patient {pid}. "
                f"The rate of disease progression appears stable across forecast "
                f"horizons, without evidence of accelerating worsening."
            )

        # Cohort context
        dist = self._store.indices_distribution(fold, "trajectory_acceleration_flag")
        if dist and dist.get("n", 0) > 0:
            accel_rate = dist["mean"]
            clinical += (
                f" In this cohort, {accel_rate:.0%} of patients show detected acceleration."
            )

        technical = (
            f"trajectory_acceleration_flag = {int(accel)} from indices_ui.csv. "
            f"Defined as 1 if the predictive mean deterioration rate increases monotonically "
            f"across 180/365/730-day horizons with >5% total delta."
        )

        return ReasonerAnswer(clinical, technical, evidence, status, guidance)

    def _handle_trajectory(self, fold: str, pid: int, question: str) -> ReasonerAnswer:
        idx, demo, visit, status, (subj, obj, poss) = self._patient_context(fold, pid)
        lat_res = self._store.get_patient_latents(fold, pid)
        evidence = [lat_res.pointer, self._store.get_patient_indices(fold, pid).pointer]
        df = lat_res.data
        guidance = self._build_guidance(idx)

        if df is None or df.empty:
            return ReasonerAnswer(
                f"No trajectory data is available for patient {pid}. "
                f"The model needs at least 2 clinic visits to estimate a disease trajectory.",
                "No rows in latent_state_trajectories_test.csv for this patient.",
                evidence, "additional_data_needed", guidance,
            )

        last = df.iloc[-1]
        xm = last.get("xM_mean", np.nan)
        xc = last.get("xC_mean", np.nan)
        xn = last.get("xN_mean", np.nan)
        fu_days = df["t_days_since_baseline"].max()
        fu_yrs = fu_days / 365.25 if fu_days else 0

        parts = []
        for label, val, col, thresholds in [
            ("motor", xm, "xM_last", (0.2, 0.4, 0.6)),
            ("cognitive", xc, "xC_last", (0.15, 0.3, 0.5)),
            ("autonomic", xn, "xN_last", (0.2, 0.4, 0.6)),
        ]:
            if isinstance(val, float) and not np.isnan(val):
                sev = self._severity_label(val, thresholds)
                pct = self._store.indices_percentile(fold, col, val)
                pctx = f" (percentile {pct:.0f})" if pct is not None else ""
                parts.append(f"{label} burden {val:.3f} ({sev}{pctx})")

        vals_valid = {
            "motor": xm, "cognitive": xc, "autonomic": xn
        }
        vals_clean = {k: v for k, v in vals_valid.items()
                      if isinstance(v, float) and not np.isnan(v)}
        dominant = max(vals_clean, key=vals_clean.get) if vals_clean else None

        clinical = (
            f"Patient {pid} has been followed for {fu_yrs:.1f} years across "
            f"{len(df)} visits. Latest estimated disease state: {'; '.join(parts)}."
        )
        if dominant:
            clinical += f" The {dominant} domain currently shows the highest burden."

        technical = (
            f"Latent states from latent_state_trajectories_test.csv, last visit row. "
            f"xM/xC/xN are posterior means in (0,1) with monotone non-decreasing constraint. "
            f"Severity labels based on thresholds: mild <0.2/0.15, moderate <0.4/0.3, "
            f"substantial <0.6/0.5, severe >=0.6/0.5."
        )

        return ReasonerAnswer(clinical, technical, evidence, status, guidance)

    def _handle_prediction_status(self, fold: str, pid: int,
                                   question: str) -> ReasonerAnswer:
        idx, demo, visit, status, (subj, obj, poss) = self._patient_context(fold, pid)
        evidence = [self._store.get_patient_indices(fold, pid).pointer]
        guidance = self._build_guidance(idx)

        if idx is None:
            return ReasonerAnswer(
                f"No data found for patient {pid} in this cohort.",
                "Patient not in indices_ui.csv.", evidence,
                "additional_data_needed", guidance,
            )

        conf = idx.get("confidence_flag")
        supp = idx.get("suppressed_at_last_anchor")
        reason = str(idx.get("suppression_reason_last_anchor", ""))
        n_vis = idx.get("n_visits")
        anchor_complete = idx.get("anchor_complete_all3_last_anchor")
        delta_iqr = idx.get("delta_latent_iqr_last_anchor")
        tau = idx.get("tau_conf_last_anchor")
        passed = idx.get("passed_conditional_runs_last_anchor")
        l_cond = idx.get("L_cond_last_anchor")

        is_ready = (conf == 1 and supp == 0)

        if is_ready:
            clinical = (
                f"Personalised predictions are available and considered reliable for patient {pid}. "
                f"The model conditioned successfully at {poss} last anchor visit "
                f"({_cfmt(passed)}/{_cfmt(l_cond)} conditioning runs passed), "
                f"the latent state uncertainty ({_cfmt(delta_iqr, precision=4)}) is within "
                f"the confidence threshold ({_cfmt(tau, precision=2)}), "
                f"and the anchor visit had complete assessments."
            )
        else:
            clinical = (
                f"Personalised predictions are not yet available for patient {pid}. "
                f"{guidance}"
            )
            clinical += f" Currently, {subj} has {n_vis} recorded visit(s)"
            if anchor_complete is not None:
                clinical += (
                    f", the last anchor is {'complete' if int(anchor_complete) == 1 else 'incomplete'}"
                )
            clinical += "."

        technical = (
            f"confidence_flag={conf}, suppressed_at_last_anchor={supp}, "
            f"suppression_reason='{reason}', n_visits={n_vis}, "
            f"anchor_complete_all3={anchor_complete}, "
            f"delta_latent_iqr={delta_iqr}, tau_conf={tau}, "
            f"passed_runs={passed}/{l_cond}."
        )

        return ReasonerAnswer(clinical, technical, evidence, status, guidance)

    def _handle_calibration(self, fold: str, pid: int, question: str) -> ReasonerAnswer:
        cal_res = self._store.get_threshold_calibration(fold)
        evidence = [cal_res.pointer]
        cal = cal_res.data

        if cal.empty:
            return ReasonerAnswer(
                "Calibration data is not available for this cohort.",
                "threshold_event_calibration.csv is empty.",
                evidence, "not_applicable", "",
            )

        # Summarise
        rows = []
        for _, r in cal.iterrows():
            ev = _EVENT_LABELS.get(r.get("event", ""), r.get("event", ""))
            task = r.get("task", "")
            brier = r.get("brier_score", np.nan)
            bss = r.get("brier_skill_score", np.nan)
            ece = r.get("expected_calibration_error", np.nan)
            rows.append(f"{ev} ({task}): Brier={brier:.4f}, BSS={bss:.4f}, ECE={ece:.4f}")

        # Overall assessment
        mean_ece = cal["expected_calibration_error"].mean()
        mean_bss = cal["brier_skill_score"].mean()

        if mean_ece < 0.05 and mean_bss > 0:
            quality = "well-calibrated overall, with predicted probabilities closely matching observed event rates"
        elif mean_ece < 0.05:
            quality = "reasonably calibrated, though the Brier skill score does not consistently exceed the base-rate benchmark"
        else:
            quality = "showing some calibration gaps that should be considered when interpreting individual risk estimates"

        clinical = (
            f"Threshold event calibration for this cohort: the model is {quality}. "
            f"Mean calibration error (ECE) across all events and tasks is {mean_ece:.4f}. "
            f"Mean Brier skill score is {mean_bss:.4f} (positive = better than base rate)."
        )

        technical = "Per-event detail:\n" + "\n".join(rows)

        return ReasonerAnswer(clinical, technical, evidence, "not_applicable", "")

    def _handle_model_performance(self, fold: str, pid: int,
                                   question: str) -> ReasonerAnswer:
        agg_res = self._store.get_aggregate_metrics(fold)
        evidence = [agg_res.pointer]
        agg = agg_res.data

        if agg.empty:
            return ReasonerAnswer(
                "Aggregate metrics are not available for this cohort.",
                "aggregate_metrics.csv is empty.", evidence, "not_applicable", "",
            )

        # Filter to reliable rows
        if "reliable" in agg.columns:
            reliable = agg[agg["reliable"].astype(str).str.lower().isin(["true", "1"])]
        else:
            reliable = agg

        if reliable.empty:
            reliable = agg

        # Summary by endpoint
        parts = []
        for ep, ep_label in [("updrs3", "UPDRS-III (motor)"),
                             ("moca", "MoCA (cognitive)"),
                             ("scopa_aut", "SCOPA-AUT (autonomic)")]:
            sub = reliable[reliable["endpoint"].str.lower().str.contains(ep)]
            if sub.empty:
                continue
            mae_range = f"{sub['MAE'].min():.2f}\u2013{sub['MAE'].max():.2f}"
            cov95 = sub["coverage95"].mean() if "coverage95" in sub.columns else np.nan
            parts.append(f"{ep_label}: MAE {mae_range}, mean 95% coverage {cov95:.1%}")

        clinical = (
            f"Model forecast accuracy for this cohort: {'; '.join(parts)}. "
        )

        # Endpoint-specific context
        if any(reliable["endpoint"].str.contains("updrs", case=False)):
            updrs_mae = reliable[reliable["endpoint"].str.contains("updrs", case=False)]["MAE"].mean()
            clinical += (
                f"For context, the UPDRS-III scale ranges from 0 to 132 and a clinically "
                f"meaningful change is typically considered to be 3\u20135 points; "
                f"the model's average MAE of {updrs_mae:.1f} points is within this range."
            )

        technical = (
            "From aggregate_metrics.csv (reliable rows only if available). "
            "MAE = mean absolute error, CRPS = continuous ranked probability score, "
            "coverage95 = fraction of observed values within the 95% prediction interval."
        )

        return ReasonerAnswer(clinical, technical, evidence, "not_applicable", "")

    def _handle_convergence(self, fold: str, pid: int, question: str) -> ReasonerAnswer:
        conv_res = self._store.get_convergence(fold)
        evidence = [conv_res.pointer]
        gs = conv_res.data.get("global_summary", {})

        met = gs.get("convergence_met", None)
        rhat = gs.get("max_r_hat")
        ess_bulk = gs.get("min_ess_bulk")
        ess_tail = gs.get("min_ess_tail")
        divs = gs.get("total_post_warmup_divergences")

        if met:
            clinical = (
                f"MCMC inference for this cohort has converged successfully. "
                f"All chains agree (max R-hat = {rhat}), effective sample sizes are adequate "
                f"(minimum bulk ESS = {ess_bulk}, minimum tail ESS = {ess_tail}), "
                f"and there were {divs} post-warmup divergent transitions. "
                f"These diagnostics indicate that the posterior estimates are reliable."
            )
        else:
            clinical = (
                f"Convergence criteria were not fully met for this cohort. "
                f"Max R-hat = {rhat} (target < 1.01), min ESS bulk = {ess_bulk}, "
                f"divergences = {divs}. Results from this cohort should be interpreted "
                f"with additional caution."
            )

        technical = (
            f"From convergence_diagnostics.json global_summary. "
            f"NUTS sampler with 4 chains, 1000 warmup + 1000 sampling iterations, "
            f"thinning by 5, retaining 200 draws per chain."
        )

        return ReasonerAnswer(clinical, technical, evidence, "not_applicable", "")

    def _handle_reviewer(self, fold: str, pid: int, question: str) -> ReasonerAnswer:
        evidence: List[EvidencePointer] = []

        # Convergence
        conv_res = self._store.get_convergence(fold)
        evidence.append(conv_res.pointer)
        gs = conv_res.data.get("global_summary", {})
        conv_ok = gs.get("convergence_met", False)

        # PPC
        ppc_res = self._store.get_ppc_summary(fold)
        evidence.append(ppc_res.pointer)
        ppc = ppc_res.data
        ppc_flags = []
        pit = ppc.get("pit_calibration", {})
        for ep, vals in pit.items():
            if vals.get("flag_ks_gt_0p10", 0) == 1:
                ppc_flags.append(f"PIT KS flag on {ep} (KS={vals.get('ks_statistic', '?')})")
        residual = ppc.get("residual_cross_endpoint_dependence", {})
        if residual.get("flagged", 0) == 1:
            ppc_flags.append(
                f"Residual cross-endpoint dependence flagged (Frobenius={residual.get('frobenius_off_diagonal', '?')})"
            )

        # Calibration
        cal_res = self._store.get_threshold_calibration(fold)
        evidence.append(cal_res.pointer)
        cal = cal_res.data
        mean_ece = cal["expected_calibration_error"].mean() if not cal.empty else np.nan

        # Aggregate metrics
        agg_res = self._store.get_aggregate_metrics(fold)
        evidence.append(agg_res.pointer)

        # Assessment
        issues = []
        if not conv_ok:
            issues.append("convergence criteria not met")
        if ppc_flags:
            issues.extend(ppc_flags)
        if isinstance(mean_ece, float) and not np.isnan(mean_ece) and mean_ece > 0.10:
            issues.append(f"mean ECE above 0.10 ({mean_ece:.4f})")

        if not issues:
            clinical = (
                f"Based on available diagnostics for this cohort: "
                f"MCMC convergence criteria are met, "
                f"posterior predictive checks show {'no flags' if not ppc_flags else ', '.join(ppc_flags)}, "
                f"and threshold event calibration error is acceptable (mean ECE = {mean_ece:.4f}). "
                f"These results are consistent with a well-fitted model. "
                f"Note: this assessment is limited to the diagnostics present in the bundle "
                f"and does not substitute for domain-expert review of clinical plausibility."
            )
        else:
            clinical = (
                f"Some diagnostics warrant attention for this cohort: {'; '.join(issues)}. "
                f"A reviewer should examine these points in detail. "
                f"Other diagnostics (convergence={'passed' if conv_ok else 'issues noted'}, "
                f"mean ECE={mean_ece:.4f}) are within expected ranges."
            )

        technical = (
            f"Checked: convergence_diagnostics.json (convergence_met={conv_ok}), "
            f"ppc_summary.json (PIT flags: {len(ppc_flags)}), "
            f"threshold_event_calibration.csv (mean ECE={mean_ece:.4f}), "
            f"aggregate_metrics.csv."
        )

        return ReasonerAnswer(clinical, technical, evidence, "not_applicable", "")

    def _handle_comparison(self, fold: str, pid: int, question: str) -> ReasonerAnswer:
        idx, demo, visit, status, (subj, obj, poss) = self._patient_context(fold, pid)
        evidence = [self._store.get_patient_indices(fold, pid).pointer]
        guidance = self._build_guidance(idx)

        if idx is None:
            return ReasonerAnswer(
                f"No data for patient {pid} in this cohort.",
                "Not in indices_ui.csv.", evidence, "additional_data_needed", guidance,
            )

        comparisons = []
        for col, label, unit, precision in [
            ("xM_last", "motor burden", "", 3),
            ("xC_last", "cognitive burden", "", 3),
            ("xN_last", "autonomic burden", "", 3),
            ("motor_volatility_index", "motor volatility", " UPDRS points", 1),
            ("near_term_cognitive_fragility_index", "cognitive fragility", "", 3),
        ]:
            val = idx.get(col)
            if val is None or (isinstance(val, float) and np.isnan(val)):
                comparisons.append(f"{label}: not yet computable")
                continue
            dist = self._store.indices_distribution(fold, col)
            pct = self._store.indices_percentile(fold, col, float(val))
            if dist and pct is not None:
                comparisons.append(
                    f"{label}: {val:.{precision}f}{unit} "
                    f"(percentile {pct:.0f}, "
                    f"cohort median {dist['median']:.{precision}f}{unit})"
                )
            else:
                comparisons.append(f"{label}: {val:.{precision}f}{unit}")

        # Acceleration prevalence
        accel = idx.get("trajectory_acceleration_flag")
        accel_dist = self._store.indices_distribution(fold, "trajectory_acceleration_flag")
        if accel is not None and not (isinstance(accel, float) and np.isnan(accel)):
            accel_txt = "detected" if int(accel) == 1 else "not detected"
            if accel_dist and accel_dist.get("n", 0) > 0:
                rate = accel_dist["mean"]
                accel_txt += f" ({rate:.0%} of cohort shows acceleration)"
            comparisons.append(f"trajectory acceleration: {accel_txt}")

        clinical = (
            f"Patient {pid} compared to this validation cohort: {'; '.join(comparisons)}."
        )

        technical = (
            "Percentiles computed from all patients in the same fold's indices_ui.csv. "
            "Distribution statistics: median and IQR computed on non-missing values."
        )

        return ReasonerAnswer(clinical, technical, evidence, status, guidance)

    def _handle_patient_summary(self, fold: str, pid: int,
                                 question: str) -> ReasonerAnswer:
        idx, demo, visit, status, (subj, obj, poss) = self._patient_context(fold, pid)
        evidence = [self._store.get_patient_indices(fold, pid).pointer]
        if demo:
            evidence.append(EvidencePointer("01_splits/fold_assignments.csv",
                                            {"patient_id": pid}))
        guidance = self._build_guidance(idx)

        if idx is None:
            return ReasonerAnswer(
                f"No data found for patient {pid} in this cohort.",
                "Not in indices_ui.csv.", evidence, "additional_data_needed", guidance,
            )

        sex = _sex_label(demo.get("sex"))
        age = demo.get("age_baseline_years")
        cohort = demo.get("cohort", "")
        n_vis = idx.get("n_visits")
        fu = idx.get("follow_up_days")

        parts = []
        parts.append(f"Patient {pid} is a {age:.0f}-year-old {sex.lower()}" if isinstance(age, (int, float)) else f"Patient {pid}")
        if cohort:
            parts[0] += f" in the {cohort} cohort"
        if isinstance(n_vis, (int, float)) and isinstance(fu, (int, float)):
            parts.append(f"followed for {fu/365.25:.1f} years across {int(n_vis)} visits")

        xm = idx.get("xM_last")
        xc = idx.get("xC_last")
        xn = idx.get("xN_last")
        if all(isinstance(v, float) and not np.isnan(v) for v in [xm, xc, xn]):
            parts.append(
                f"latest disease state: motor={xm:.3f}, "
                f"cognitive={xc:.3f}, autonomic={xn:.3f}"
            )

        is_ready = (idx.get("confidence_flag") == 1 and idx.get("suppressed_at_last_anchor") == 0)
        if is_ready:
            parts.append("personalised predictions are available")
        else:
            parts.append("additional clinical data needed before personalised predictions can be generated")

        clinical = ", ".join(parts) + "."

        technical = (
            f"Compiled from indices_ui.csv and fold_assignments.csv. "
            f"n_visits={n_vis}, follow_up_days={fu}, "
            f"confidence_flag={idx.get('confidence_flag')}, "
            f"suppressed_at_last_anchor={idx.get('suppressed_at_last_anchor')}."
        )

        return ReasonerAnswer(clinical, technical, evidence, status, guidance)

    # ------------------------------------------------------------------
    # Main entry point
    # ------------------------------------------------------------------
    def ask(self, question: str, fold: str, patient_id: int) -> ReasonerAnswer:
        """
        Parse a free-text question and return a grounded answer.
        All answers are deterministic and reference specific source files.
        """
        q = question.strip()
        if not q:
            return ReasonerAnswer(
                "Please enter a question.",
                "No question provided.", [], "not_applicable", "",
            )

        # Detect intent
        detected_intent = "patient_summary"  # fallback
        for intent_name, pattern, _ in self._intent_rules:
            if pattern.search(q):
                detected_intent = intent_name
                break

        # Route to handler
        handlers = {
            "risk_updrs10": lambda: self._handle_risk_event(
                fold, patient_id, "dUPDRS>=10", "Motor worsening (UPDRS +10)", q),
            "risk_updrs5": lambda: self._handle_risk_event(
                fold, patient_id, "dUPDRS>=5", "Motor worsening (UPDRS +5)", q),
            "risk_moca2": lambda: self._handle_risk_event(
                fold, patient_id, "dMoCA<=-2", "Cognitive decline (MoCA -2)", q),
            "risk_moca3": lambda: self._handle_risk_event(
                fold, patient_id, "dMoCA<=-3", "Cognitive decline (MoCA -3)", q),
            "risk_general": lambda: self._handle_risk_event(
                fold, patient_id, "dUPDRS>=10", "Motor worsening (UPDRS +10)", q),
            "cognitive_fragility": lambda: self._handle_cognitive_fragility(fold, patient_id, q),
            "motor_volatility": lambda: self._handle_motor_volatility(fold, patient_id, q),
            "trajectory_acceleration": lambda: self._handle_trajectory_acceleration(fold, patient_id, q),
            "trajectory": lambda: self._handle_trajectory(fold, patient_id, q),
            "prediction_status": lambda: self._handle_prediction_status(fold, patient_id, q),
            "calibration": lambda: self._handle_calibration(fold, patient_id, q),
            "model_performance": lambda: self._handle_model_performance(fold, patient_id, q),
            "convergence": lambda: self._handle_convergence(fold, patient_id, q),
            "reviewer": lambda: self._handle_reviewer(fold, patient_id, q),
            "comparison": lambda: self._handle_comparison(fold, patient_id, q),
            "patient_summary": lambda: self._handle_patient_summary(fold, patient_id, q),
            "horizon_specific": lambda: self._handle_risk_event(
                fold, patient_id, "dUPDRS>=10", "Motor worsening (UPDRS +10)", q),
        }

        handler = handlers.get(detected_intent, handlers["patient_summary"])
        return handler()


# Instantiate
REASONER = ClinicalReasoner(STORE)



In [ ]:
%%capture _console_log
# @title D3. Clinical reasoner console setup 

# ============================================================================
# Insight card definitions
# ============================================================================
@dataclass
class InsightCard:
    key: str
    title: str
    subtitle: str
    icon_color: str

_INSIGHT_CARDS = [
    InsightCard(
        "comprehensive_profile",
        "Comprehensive clinical profile",
        "Full narrative: demographics, disease state, trajectory pattern, and current outlook",
        "#0f3460",
    ),
    InsightCard(
        "motor_outlook",
        "Motor progression outlook",
        "Motor risk across horizons, volatility of predictions, and trajectory dynamics",
        "#2166ac",
    ),
    InsightCard(
        "cognitive_vulnerability",
        "Cognitive vulnerability assessment",
        "Near-term cognitive decline risk, fragility index, and protective or risk factors",
        "#b2182b",
    ),
    InsightCard(
        "multi_domain_landscape",
        "Multi-domain risk landscape",
        "Simultaneous view of motor, cognitive and autonomic risks with domain prioritisation",
        "#4dac26",
    ),
    InsightCard(
        "cohort_benchmarking",
        "Cohort benchmarking",
        "Where this patient stands relative to the full validation cohort across all indices",
        "#7570b3",
    ),
    InsightCard(
        "trajectory_dynamics",
        "Trajectory dynamics and acceleration",
        "Whether disease progression is stable, accelerating, or showing domain divergence",
        "#e6ab02",
    ),
    InsightCard(
        "data_completeness",
        "Data completeness and prediction readiness",
        "What clinical data is available, what is missing, and how to improve predictions",
        "#1565c0",
    ),
    InsightCard(
        "model_trustworthiness",
        "Model trustworthiness for this patient",
        "Convergence, calibration, and confidence assessment specific to this individual",
        "#666666",
    ),
]

# ============================================================================
# Insight generators (rich clinical narratives)
# ============================================================================
def _insight_comprehensive_profile(fold: str, pid: int) -> None:
    idx_res = STORE.get_patient_indices(fold, pid)
    if idx_res.data is None:
        print(f"Patient {pid} not found in this validation cohort.")
        return
    idx = idx_res.data
    demo = _get_demographics(pid)
    visit = _get_visit_summary(pid)
    lat_res = STORE.get_patient_latents(fold, pid)
    lat = lat_res.data

    sex = _sex_label(demo.get("sex"))
    age = demo.get("age_baseline_years")
    cohort = demo.get("cohort", "")
    subj, obj, poss = _pronoun(sex)
    n_vis = idx.get("n_visits")
    fu = idx.get("follow_up_days")
    fu_yrs = fu / 365.25 if isinstance(fu, (int, float)) else None

    # Opening narrative
    age_s = f"{age:.0f}-year-old" if isinstance(age, (int, float)) else ""
    para1 = f"Patient {pid} is a {age_s} {sex.lower()}"
    if cohort:
        para1 += f" enrolled in the {cohort} cohort"
    if isinstance(n_vis, (int, float)) and fu_yrs:
        para1 += f", followed across {int(n_vis)} clinic visits over {fu_yrs:.1f} years"
    para1 += "."

    # Omics
    has_omics = demo.get("has_omics")
    if has_omics == 1:
        para1 += f" Genetic and omics data are available for this participant."

    # Observation completeness
    n_updrs = visit.get("n_updrs_observed")
    n_moca = visit.get("n_moca_observed")
    n_scopa = visit.get("n_scopa_observed")
    if isinstance(n_vis, (int, float)) and n_vis > 0:
        completeness_parts = []
        for label, obs in [("motor (UPDRS-III)", n_updrs),
                           ("cognitive (MoCA)", n_moca),
                           ("autonomic (SCOPA-AUT)", n_scopa)]:
            if isinstance(obs, (int, float)):
                completeness_parts.append(f"{label} {int(obs)}/{int(n_vis)} ({obs/n_vis:.0%})")
        if completeness_parts:
            para1 += f" Assessment completeness: {', '.join(completeness_parts)}."

    # Disease state
    xm = idx.get("xM_last")
    xc = idx.get("xC_last")
    xn = idx.get("xN_last")

    para2 = ""
    if all(isinstance(v, float) and not np.isnan(v) for v in [xm, xc, xn]):
        vals = {"motor": (xm, "xM_last", (0.2, 0.4, 0.6)),
                "cognitive": (xc, "xC_last", (0.15, 0.3, 0.5)),
                "autonomic": (xn, "xN_last", (0.2, 0.4, 0.6))}
        dominant = max(vals, key=lambda k: vals[k][0])
        dominant_val = vals[dominant][0]

        domain_descs = []
        for domain, (val, col, thresh) in vals.items():
            sev = _severity_word(val, thresh)
            pct = STORE.indices_percentile(fold, col, val)
            desc = f"{domain} burden is {sev} ({val:.3f}"
            if pct is not None:
                desc += f", percentile {pct:.0f}"
            desc += ")"
            domain_descs.append(desc)

        para2 = (
            f"At {poss} most recent assessment, {', '.join(domain_descs)}. "
            f"The {dominant} domain currently dominates the clinical picture."
        )

        # Trajectory shape if available
        if lat is not None and len(lat) >= 3:
            first_xm = lat.iloc[0].get("xM_mean", np.nan)
            if isinstance(first_xm, float) and not np.isnan(first_xm) and isinstance(xm, float):
                delta_m = xm - first_xm
                if delta_m > 0.15:
                    para2 += f" Motor burden has progressed notably (+{delta_m:.3f}) since baseline."
                elif delta_m > 0.05:
                    para2 += f" Motor burden shows gradual progression (+{delta_m:.3f}) since baseline."
                else:
                    para2 += " Motor burden has remained relatively stable since baseline."

    # Derived indices
    mv = idx.get("motor_volatility_index")
    cf = idx.get("near_term_cognitive_fragility_index")
    accel = idx.get("trajectory_acceleration_flag")

    para3_parts = []
    if isinstance(mv, float) and not np.isnan(mv):
        mv_pct = STORE.indices_percentile(fold, "motor_volatility_index", mv)
        para3_parts.append(
            f"motor volatility is {mv:.1f} UPDRS points"
            + (f" (percentile {mv_pct:.0f})" if mv_pct is not None else "")
        )
    if isinstance(cf, float) and not np.isnan(cf):
        risk_level = "elevated" if cf > 0.20 else ("moderate" if cf > 0.10 else "low")
        para3_parts.append(f"near-term cognitive fragility is {cf:.1%} ({risk_level})")
    if accel is not None and not (isinstance(accel, float) and np.isnan(accel)):
        if int(accel) == 1:
            para3_parts.append("trajectory acceleration has been detected")
        else:
            para3_parts.append("no trajectory acceleration detected")

    para3 = ""
    if para3_parts:
        para3 = "Derived indices: " + ", ".join(para3_parts) + "."

    # Prediction readiness
    conf = idx.get("confidence_flag")
    supp = idx.get("suppressed_at_last_anchor")
    reason = str(idx.get("suppression_reason_last_anchor", ""))
    is_ready = (conf == 1 and supp == 0)

    if is_ready:
        para4 = "Personalised predictions are available and considered reliable for this patient."
    else:
        guidance = _IMPROVEMENT_GUIDANCE.get(reason, "")
        para4 = f"Personalised predictions are not yet available. {guidance}"

    # Render
    full_text = f"{para1}<br><br>{para2}" if para2 else para1
    if para3:
        full_text += f"<br><br>{para3}"
    full_text += f"<br><br>{para4}"

    display(HTML(f"""
    <div style="border-left: 4px solid #0f3460; padding: 12px 16px;
                background: #f5f7fa; border-radius: 2px; margin-bottom: 8px;">
        <div style="font-size: 12px; color: #333; line-height: 1.7;">{full_text}</div>
    </div>
    """))

    evidence = [str(idx_res.pointer)]
    if demo:
        evidence.append("01_splits/fold_assignments.csv")
    if visit:
        evidence.append("02_preprocessing/visit_summaries.csv")
    if lat is not None and not lat.empty:
        evidence.append(str(lat_res.pointer))
    display(HTML(
        f"<p style='color:#999; font-size:10px;'>Evidence: {' + '.join(evidence)}</p>"
    ))


def _insight_motor_outlook(fold: str, pid: int) -> None:
    idx_res = STORE.get_patient_indices(fold, pid)
    idx = idx_res.data
    demo = _get_demographics(pid)
    sex = _sex_label(demo.get("sex"))
    subj, obj, poss = _pronoun(sex)
    evidence: List[str] = [str(idx_res.pointer)]

    if idx is None:
        print(f"Patient {pid} not found in this cohort.")
        return

    parts = []

    # Risk probabilities
    risk_res = STORE.get_patient_risk_curves(fold, pid)
    evidence.append(str(risk_res.pointer))
    rdf = risk_res.data

    if rdf is not None and not rdf.empty:
        for event, event_label, clinical_meaning in [
            ("dUPDRS>=5", "UPDRS +5", "a noticeable motor change"),
            ("dUPDRS>=10", "UPDRS +10", "a clinically significant motor worsening"),
        ]:
            ev_data = rdf[rdf["event"] == event].sort_values("horizon_days")
            if ev_data.empty:
                continue
            prob_strings = []
            for _, r in ev_data.iterrows():
                h = int(r["horizon_days"])
                p = r["prob"]
                if h <= 365:
                    prob_strings.append(f"{p:.1%} at {h} days (~{h/365.25:.1f} year)")
                else:
                    prob_strings.append(f"{p:.1%} at {h} days (~{h/365.25:.1f} years)")
            parts.append(
                f"The probability of {event_label} ({clinical_meaning}) is "
                f"{', '.join(prob_strings)}."
            )

        # Trend interpretation
        updrs10 = rdf[rdf["event"] == "dUPDRS>=10"].sort_values("horizon_days")
        if len(updrs10) >= 2:
            p_first = updrs10.iloc[0]["prob"]
            p_last = updrs10.iloc[-1]["prob"]
            if p_last > p_first * 1.5:
                parts.append(
                    "Risk accumulates notably over time, suggesting progressive motor vulnerability."
                )
            elif p_last > p_first * 1.1:
                parts.append("Risk increases gradually with longer horizons.")
            else:
                parts.append("Risk remains relatively stable across horizons.")
    else:
        guidance = _IMPROVEMENT_GUIDANCE.get(
            str(idx.get("suppression_reason_last_anchor", "")), "")
        parts.append(f"Motor risk estimates are not yet available for this patient. {guidance}")

    # Volatility
    mv = idx.get("motor_volatility_index")
    if isinstance(mv, float) and not np.isnan(mv):
        mv_dist = STORE.indices_distribution(fold, "motor_volatility_index")
        mv_pct = STORE.indices_percentile(fold, "motor_volatility_index", mv)
        interp = ""
        if mv_dist and mv_pct is not None:
            if mv_pct > 75:
                interp = (
                    f", which is in the upper quartile of the cohort (percentile {mv_pct:.0f}). "
                    f"This suggests {poss} motor trajectory is harder to predict than most patients, "
                    f"potentially reflecting high measurement noise or genuine fluctuation"
                )
            elif mv_pct < 25:
                interp = (
                    f", in the lower quartile (percentile {mv_pct:.0f}). "
                    f"This indicates relatively predictable motor scores"
                )
            else:
                interp = f" (percentile {mv_pct:.0f}, near the cohort median of {mv_dist['median']:.1f})"
        parts.append(f"Motor volatility index is {mv:.1f} UPDRS points{interp}.")

    # Acceleration
    accel = idx.get("trajectory_acceleration_flag")
    if accel is not None and not (isinstance(accel, float) and np.isnan(accel)):
        if int(accel) == 1:
            parts.append(
                "Importantly, trajectory acceleration has been detected: the rate of motor "
                "worsening appears to be increasing over time, meaning longer-term forecasts "
                "show disproportionately larger changes than shorter-term ones."
            )
        else:
            parts.append(
                "No trajectory acceleration detected: the motor progression rate appears "
                "stable, without evidence of the worsening rate itself increasing."
            )

    display(HTML(f"""
    <div style="border-left: 4px solid #2166ac; padding: 12px 16px;
                background: #f5f7fa; border-radius: 2px; margin-bottom: 8px;">
        <div style="font-size: 12px; color: #333; line-height: 1.7;">
            {'<br><br>'.join(parts)}
        </div>
    </div>
    """))
    display(HTML(f"<p style='color:#999; font-size:10px;'>Evidence: {' + '.join(evidence)}</p>"))


def _insight_cognitive_vulnerability(fold: str, pid: int) -> None:
    idx_res = STORE.get_patient_indices(fold, pid)
    idx = idx_res.data
    demo = _get_demographics(pid)
    sex = _sex_label(demo.get("sex"))
    subj, obj, poss = _pronoun(sex)
    evidence: List[str] = [str(idx_res.pointer)]

    if idx is None:
        print(f"Patient {pid} not found.")
        return

    parts = []

    # Cognitive burden
    xc = idx.get("xC_last")
    if isinstance(xc, float) and not np.isnan(xc):
        sev = _severity_word(xc, (0.15, 0.3, 0.5))
        pct = STORE.indices_percentile(fold, "xC_last", xc)
        pct_s = f" (percentile {pct:.0f})" if pct is not None else ""
        parts.append(
            f"Current cognitive burden is estimated at {xc:.3f} on the 0\u20131 latent scale, "
            f"classified as {sev}{pct_s}. This represents the model's integrated estimate "
            f"of cognitive disease state accounting for all observed MoCA scores and "
            f"their temporal pattern."
        )

    # Fragility index
    cf = idx.get("near_term_cognitive_fragility_index")
    if isinstance(cf, float) and not np.isnan(cf):
        cf_dist = STORE.indices_distribution(fold, "near_term_cognitive_fragility_index")
        cf_pct = STORE.indices_percentile(fold, "near_term_cognitive_fragility_index", cf)

        risk_word = "elevated" if cf > 0.20 else ("moderate" if cf > 0.10 else "low")
        context = ""
        if cf_dist and cf_pct is not None:
            context = (
                f" Within this cohort, {subj} ranks at percentile {cf_pct:.0f} "
                f"(cohort median: {cf_dist['median']:.1%}, "
                f"IQR: {cf_dist['q25']:.1%}\u2013{cf_dist['q75']:.1%})."
            )
        parts.append(
            f"The near-term cognitive fragility index is {cf:.1%}, indicating {risk_word} "
            f"risk of a clinically meaningful MoCA decline (2 or more points) in the "
            f"near term. This index captures the peak probability of cognitive worsening "
            f"across available forecast horizons.{context}"
        )

        if cf > 0.20:
            parts.append(
                "At this level, closer monitoring of cognitive function and consideration "
                "of cognitive screening at shorter intervals may be warranted."
            )
    else:
        explanation = _explain_missing_index(idx, "Cognitive fragility index")
        parts.append(explanation)

    # MoCA-specific risk curves
    risk_res = STORE.get_patient_risk_curves(fold, pid)
    evidence.append(str(risk_res.pointer))
    rdf = risk_res.data
    if rdf is not None and not rdf.empty:
        for event, label in [("dMoCA<=-2", "MoCA drop of 2+ points"),
                             ("dMoCA<=-3", "MoCA drop of 3+ points")]:
            ev_data = rdf[rdf["event"] == event].sort_values("horizon_days")
            if not ev_data.empty:
                h_probs = [f"{r['prob']:.1%} at {int(r['horizon_days'])}d"
                           for _, r in ev_data.iterrows()]
                parts.append(f"Probability of {label}: {', '.join(h_probs)}.")

    # Relationship to motor
    xm = idx.get("xM_last")
    if (isinstance(xc, float) and not np.isnan(xc) and
        isinstance(xm, float) and not np.isnan(xm)):
        if xc > xm:
            parts.append(
                f"Notably, cognitive burden ({xc:.3f}) currently exceeds motor burden "
                f"({xm:.3f}), suggesting a cognitive-predominant phenotype at this stage."
            )
        elif xm > xc * 2:
            parts.append(
                f"Motor burden ({xm:.3f}) substantially exceeds cognitive burden ({xc:.3f}), "
                f"suggesting cognition is relatively preserved compared to motor function."
            )

    display(HTML(f"""
    <div style="border-left: 4px solid #b2182b; padding: 12px 16px;
                background: #f5f7fa; border-radius: 2px; margin-bottom: 8px;">
        <div style="font-size: 12px; color: #333; line-height: 1.7;">
            {'<br><br>'.join(parts)}
        </div>
    </div>
    """))
    display(HTML(f"<p style='color:#999; font-size:10px;'>Evidence: {' + '.join(evidence)}</p>"))


def _insight_multi_domain(fold: str, pid: int) -> None:
    idx_res = STORE.get_patient_indices(fold, pid)
    idx = idx_res.data
    demo = _get_demographics(pid)
    sex = _sex_label(demo.get("sex"))
    subj, obj, poss = _pronoun(sex)
    evidence: List[str] = [str(idx_res.pointer)]

    if idx is None:
        print(f"Patient {pid} not found.")
        return

    parts = []

    # All three domains side by side
    xm = idx.get("xM_last")
    xc = idx.get("xC_last")
    xn = idx.get("xN_last")

    domain_data = {}
    for label, val, col, thresh in [
        ("Motor", xm, "xM_last", (0.2, 0.4, 0.6)),
        ("Cognitive", xc, "xC_last", (0.15, 0.3, 0.5)),
        ("Autonomic", xn, "xN_last", (0.2, 0.4, 0.6)),
    ]:
        if isinstance(val, float) and not np.isnan(val):
            sev = _severity_word(val, thresh)
            pct = STORE.indices_percentile(fold, col, val)
            domain_data[label.lower()] = {"val": val, "sev": sev, "pct": pct}

    if domain_data:
        sorted_domains = sorted(domain_data.items(), key=lambda x: x[1]["val"], reverse=True)
        domain_desc = []
        for dname, dd in sorted_domains:
            pct_s = f", percentile {dd['pct']:.0f}" if dd["pct"] is not None else ""
            domain_desc.append(f"{dname} {dd['val']:.3f} ({dd['sev']}{pct_s})")

        parts.append(
            f"Current disease burden across domains, ordered by severity: {', '.join(domain_desc)}."
        )

        # Domain discordance analysis
        if len(sorted_domains) >= 2:
            highest = sorted_domains[0]
            lowest = sorted_domains[-1]
            ratio = highest[1]["val"] / max(lowest[1]["val"], 0.001)
            if ratio > 3:
                parts.append(
                    f"There is substantial domain discordance: {highest[0]} burden is "
                    f"{ratio:.1f}x higher than {lowest[0]}. This asymmetry may reflect "
                    f"a phenotype where {highest[0].lower()} impairment is the primary "
                    f"driver of disease impact."
                )
            elif ratio > 1.5:
                parts.append(
                    f"{highest[0]} burden leads the clinical picture, though other domains "
                    f"are also engaged."
                )
            else:
                parts.append(
                    "Disease burden is relatively balanced across domains, without a single "
                    "domain clearly dominating."
                )

    # Risk landscape across all events
    risk_res = STORE.get_patient_risk_curves(fold, pid)
    evidence.append(str(risk_res.pointer))
    rdf = risk_res.data

    if rdf is not None and not rdf.empty:
        # Find the single highest risk
        max_row = rdf.loc[rdf["prob"].idxmax()]
        max_event = _EVENT_LABELS.get(max_row["event"], max_row["event"])
        max_h = int(max_row["horizon_days"])
        max_p = max_row["prob"]

        parts.append(
            f"Across all four tracked clinical events and all horizons, the highest "
            f"predicted probability is {max_p:.1%} for {max_event.lower()} at "
            f"{max_h} days (~{max_h/365.25:.1f} years). "
        )

        # Compare motor vs cognitive risk
        motor_max = rdf[rdf["event"].str.contains("UPDRS")]["prob"].max()
        cog_max = rdf[rdf["event"].str.contains("MoCA")]["prob"].max()
        if isinstance(motor_max, float) and isinstance(cog_max, float):
            if motor_max > cog_max * 1.5:
                parts.append(
                    f"Motor event risk (peak {motor_max:.1%}) exceeds cognitive event "
                    f"risk (peak {cog_max:.1%}), suggesting motor worsening is the more "
                    f"immediate clinical concern."
                )
            elif cog_max > motor_max * 1.5:
                parts.append(
                    f"Cognitive event risk (peak {cog_max:.1%}) exceeds motor event "
                    f"risk (peak {motor_max:.1%}), highlighting cognitive vulnerability "
                    f"as a priority for monitoring."
                )
            else:
                parts.append(
                    f"Motor (peak {motor_max:.1%}) and cognitive (peak {cog_max:.1%}) "
                    f"event risks are comparable, warranting balanced monitoring across domains."
                )
    else:
        guidance = _IMPROVEMENT_GUIDANCE.get(
            str(idx.get("suppression_reason_last_anchor", "")), "")
        parts.append(f"Risk estimates are not yet available for this patient. {guidance}")

    display(HTML(f"""
    <div style="border-left: 4px solid #4dac26; padding: 12px 16px;
                background: #f5f7fa; border-radius: 2px; margin-bottom: 8px;">
        <div style="font-size: 12px; color: #333; line-height: 1.7;">
            {'<br><br>'.join(parts)}
        </div>
    </div>
    """))
    display(HTML(f"<p style='color:#999; font-size:10px;'>Evidence: {' + '.join(evidence)}</p>"))


def _insight_cohort_benchmarking(fold: str, pid: int) -> None:
    idx_res = STORE.get_patient_indices(fold, pid)
    idx = idx_res.data
    demo = _get_demographics(pid)
    sex = _sex_label(demo.get("sex"))
    subj, obj, poss = _pronoun(sex)
    evidence: List[str] = [str(idx_res.pointer)]

    if idx is None:
        print(f"Patient {pid} not found.")
        return

    parts = []

    benchmarks = []
    for col, label, unit, precision, higher_is_worse in [
        ("xM_last", "Motor burden", "", 3, True),
        ("xC_last", "Cognitive burden", "", 3, True),
        ("xN_last", "Autonomic burden", "", 3, True),
        ("motor_volatility_index", "Motor volatility", " UPDRS pts", 1, True),
        ("near_term_cognitive_fragility_index", "Cognitive fragility", "", 3, True),
    ]:
        val = idx.get(col)
        if val is None or (isinstance(val, float) and np.isnan(val)):
            benchmarks.append({"label": label, "available": False})
            continue
        dist = STORE.indices_distribution(fold, col)
        pct = STORE.indices_percentile(fold, col, float(val))
        benchmarks.append({
            "label": label, "available": True, "val": val,
            "pct": pct, "dist": dist, "unit": unit,
            "precision": precision, "higher_is_worse": higher_is_worse,
        })

    # Narrative for available benchmarks
    above_median = []
    below_median = []
    for b in benchmarks:
        if not b["available"] or b.get("pct") is None:
            continue
        prec = b["precision"]
        val_s = f"{b['val']:.{prec}f}{b['unit']}"
        median_s = f"{b['dist']['median']:.{prec}f}{b['unit']}" if b.get("dist") else ""
        entry = f"{b['label']} {val_s} (percentile {b['pct']:.0f}, cohort median {median_s})"
        if b["pct"] >= 50:
            above_median.append(entry)
        else:
            below_median.append(entry)

    if above_median:
        word = "worse" if benchmarks[0].get("higher_is_worse", True) else "higher"
        parts.append(
            f"Above the cohort median ({word} than the typical patient): "
            + ", ".join(above_median) + "."
        )
    if below_median:
        word = "better" if benchmarks[0].get("higher_is_worse", True) else "lower"
        parts.append(
            f"Below the cohort median ({word} than the typical patient): "
            + ", ".join(below_median) + "."
        )

    unavail = [b["label"] for b in benchmarks if not b["available"]]
    if unavail:
        parts.append(
            f"Not yet computable for this patient: {', '.join(unavail)}. "
            + _explain_missing_index(idx, "These indices")
        )

    # Acceleration in cohort context
    accel = idx.get("trajectory_acceleration_flag")
    accel_dist = STORE.indices_distribution(fold, "trajectory_acceleration_flag")
    if accel is not None and not (isinstance(accel, float) and np.isnan(accel)):
        accel_txt = "detected" if int(accel) == 1 else "not detected"
        cohort_rate = ""
        if accel_dist and accel_dist.get("n", 0) > 0:
            cohort_rate = f" ({accel_dist['mean']:.0%} of patients in this cohort show acceleration)"
        parts.append(f"Trajectory acceleration: {accel_txt}{cohort_rate}.")

    # Overall positioning
    avail_pcts = [b["pct"] for b in benchmarks if b["available"] and b.get("pct") is not None]
    if avail_pcts:
        mean_pct = np.mean(avail_pcts)
        if mean_pct > 75:
            parts.append(
                f"Overall, this patient's average percentile across available indices is "
                f"{mean_pct:.0f}, placing {obj} in the more severely affected quarter of the cohort."
            )
        elif mean_pct > 50:
            parts.append(
                f"Overall average percentile is {mean_pct:.0f}, indicating disease burden "
                f"above the cohort median but not in the most affected quartile."
            )
        elif mean_pct > 25:
            parts.append(
                f"Overall average percentile is {mean_pct:.0f}, indicating disease burden "
                f"below the cohort median."
            )
        else:
            parts.append(
                f"Overall average percentile is {mean_pct:.0f}, placing {obj} in the "
                f"least affected quarter of the cohort."
            )

    display(HTML(f"""
    <div style="border-left: 4px solid #7570b3; padding: 12px 16px;
                background: #f5f7fa; border-radius: 2px; margin-bottom: 8px;">
        <div style="font-size: 12px; color: #333; line-height: 1.7;">
            {'<br><br>'.join(parts)}
        </div>
    </div>
    """))
    display(HTML(f"<p style='color:#999; font-size:10px;'>Evidence: {' + '.join(evidence)}</p>"))


def _insight_trajectory_dynamics(fold: str, pid: int) -> None:
    idx_res = STORE.get_patient_indices(fold, pid)
    idx = idx_res.data
    lat_res = STORE.get_patient_latents(fold, pid)
    lat = lat_res.data
    demo = _get_demographics(pid)
    sex = _sex_label(demo.get("sex"))
    subj, obj, poss = _pronoun(sex)
    evidence: List[str] = [str(idx_res.pointer), str(lat_res.pointer)]

    if idx is None:
        print(f"Patient {pid} not found.")
        return

    parts = []

    if lat is None or lat.empty or len(lat) < 2:
        n_vis = idx.get("n_visits")
        parts.append(
            f"Trajectory dynamics cannot be assessed for patient {pid} because "
            + (f"only {int(n_vis)} visit is available. " if isinstance(n_vis, (int, float)) and n_vis < 2
               else "insufficient longitudinal data is available. ")
            + "At least 2 clinic visits are needed to estimate how the disease evolves over time."
        )
    else:
        fu_days = lat["t_days_since_baseline"].max()
        fu_yrs = fu_days / 365.25

        # Per-domain trajectory analysis
        for key, domain, col_last in [
            ("xM", "motor", "xM_last"),
            ("xC", "cognitive", "xC_last"),
            ("xN", "autonomic", "xN_last"),
        ]:
            mean_col = f"{key}_mean"
            if mean_col not in lat.columns:
                continue
            vals = lat[mean_col].dropna().values
            if len(vals) < 2:
                continue

            first_val = vals[0]
            last_val = vals[-1]
            delta = last_val - first_val
            n_pts = len(vals)

            # Characterise shape
            if delta < 0.02:
                shape = "essentially stable"
            elif delta < 0.10:
                shape = "showing slow progression"
            elif delta < 0.25:
                shape = "showing moderate progression"
            else:
                shape = "showing substantial progression"

            # Check for non-monotone appearance (even though latent states are monotone,
            # the rate can vary)
            if n_pts >= 4:
                mid = n_pts // 2
                first_half_rate = (vals[mid] - vals[0]) / max(mid, 1)
                second_half_rate = (vals[-1] - vals[mid]) / max(n_pts - mid, 1)
                if second_half_rate > first_half_rate * 2 and second_half_rate > 0.001:
                    shape += " with acceleration in the later period"
                elif first_half_rate > second_half_rate * 2 and first_half_rate > 0.001:
                    shape += " with the rate slowing in the later period"

            parts.append(
                f"The {domain} domain ({first_val:.3f} \u2192 {last_val:.3f} over "
                f"{fu_yrs:.1f} years) is {shape}."
            )

        # Domain divergence
        xm = idx.get("xM_last")
        xc = idx.get("xC_last")
        xn = idx.get("xN_last")
        first_xm = lat.iloc[0].get("xM_mean", np.nan)
        first_xc = lat.iloc[0].get("xC_mean", np.nan)
        first_xn = lat.iloc[0].get("xN_mean", np.nan)

        if all(isinstance(v, float) and not np.isnan(v)
               for v in [xm, xc, xn, first_xm, first_xc, first_xn]):
            deltas = {
                "motor": xm - first_xm,
                "cognitive": xc - first_xc,
                "autonomic": xn - first_xn,
            }
            fastest = max(deltas, key=deltas.get)
            slowest = min(deltas, key=deltas.get)
            if deltas[fastest] > deltas[slowest] * 3 and deltas[fastest] > 0.05:
                parts.append(
                    f"Domain divergence observed: {fastest} has progressed "
                    f"{deltas[fastest]:.3f} units while {slowest} has progressed only "
                    f"{deltas[slowest]:.3f}. The disease is evolving asymmetrically across domains."
                )

    # Acceleration flag
    accel = idx.get("trajectory_acceleration_flag")
    if accel is not None and not (isinstance(accel, float) and np.isnan(accel)):
        if int(accel) == 1:
            parts.append(
                "The trajectory acceleration flag is positive, confirming that "
                "the model's longer-horizon predictions show disproportionately larger "
                "worsening than shorter-horizon ones. This is consistent with the disease "
                "entering a phase of more rapid progression."
            )
        else:
            parts.append(
                "No formal trajectory acceleration detected. The progression rate, "
                "while potentially non-zero, does not show a statistically meaningful "
                "increase across forecast horizons."
            )

    display(HTML(f"""
    <div style="border-left: 4px solid #e6ab02; padding: 12px 16px;
                background: #f5f7fa; border-radius: 2px; margin-bottom: 8px;">
        <div style="font-size: 12px; color: #333; line-height: 1.7;">
            {'<br><br>'.join(parts)}
        </div>
    </div>
    """))
    display(HTML(f"<p style='color:#999; font-size:10px;'>Evidence: {' + '.join(evidence)}</p>"))


def _insight_data_completeness(fold: str, pid: int) -> None:
    idx_res = STORE.get_patient_indices(fold, pid)
    idx = idx_res.data
    demo = _get_demographics(pid)
    visit = _get_visit_summary(pid)
    sex = _sex_label(demo.get("sex"))
    subj, obj, poss = _pronoun(sex)
    evidence: List[str] = [str(idx_res.pointer)]
    if visit:
        evidence.append("02_preprocessing/visit_summaries.csv")

    if idx is None:
        print(f"Patient {pid} not found.")
        return

    parts = []
    n_vis = idx.get("n_visits")
    fu = idx.get("follow_up_days")

    # Visit profile
    if isinstance(n_vis, (int, float)) and isinstance(fu, (int, float)):
        if n_vis >= 10:
            visit_quality = "extensive longitudinal follow-up"
        elif n_vis >= 5:
            visit_quality = "moderate follow-up"
        elif n_vis >= 2:
            visit_quality = "limited but sufficient follow-up for basic conditioning"
        else:
            visit_quality = "minimal data (single visit only)"
        parts.append(
            f"Patient {pid} has {int(n_vis)} recorded visits over {fu/365.25:.1f} years, "
            f"representing {visit_quality}."
        )

    # Endpoint completeness
    n_updrs = visit.get("n_updrs_observed")
    n_moca = visit.get("n_moca_observed")
    n_scopa = visit.get("n_scopa_observed")
    n_total = visit.get("n_visits", n_vis)

    if isinstance(n_total, (int, float)) and n_total > 0:
        completeness = {}
        for label, obs in [("UPDRS-III", n_updrs), ("MoCA", n_moca), ("SCOPA-AUT", n_scopa)]:
            if isinstance(obs, (int, float)):
                completeness[label] = obs / n_total

        if completeness:
            full = [k for k, v in completeness.items() if v >= 0.9]
            partial = [f"{k} ({v:.0%})" for k, v in completeness.items() if 0.5 <= v < 0.9]
            sparse = [f"{k} ({v:.0%})" for k, v in completeness.items() if v < 0.5]

            if full:
                parts.append(f"Near-complete assessment coverage for: {', '.join(full)}.")
            if partial:
                parts.append(f"Partial coverage for: {', '.join(partial)}.")
            if sparse:
                parts.append(
                    f"Sparse coverage for: {', '.join(sparse)}. "
                    f"Low observation rates in these endpoints limit the model's ability "
                    f"to infer the corresponding latent disease dimensions precisely."
                )

    # Prediction readiness
    conf = idx.get("confidence_flag")
    supp = idx.get("suppressed_at_last_anchor")
    reason = str(idx.get("suppression_reason_last_anchor", ""))
    is_ready = (conf == 1 and supp == 0)

    if is_ready:
        passed = idx.get("passed_conditional_runs_last_anchor")
        l_cond = idx.get("L_cond_last_anchor")
        delta_iqr = idx.get("delta_latent_iqr_last_anchor")
        tau = idx.get("tau_conf_last_anchor")
        parts.append(
            f"Prediction readiness: all requirements met. The model conditioned successfully "
            f"({_cfmt(passed)}/{_cfmt(l_cond)} runs), and the latent state uncertainty "
            f"({_cfmt(delta_iqr, precision=4)}) is within the confidence threshold "
            f"({_cfmt(tau, precision=2)}). Personalised forecasts are available."
        )
    else:
        guidance = _IMPROVEMENT_GUIDANCE.get(reason, _IMPROVEMENT_GUIDANCE.get("", ""))
        parts.append(f"Prediction readiness: additional data needed. {guidance}")

        # Specific actionable advice
        actions = []
        if isinstance(n_vis, (int, float)) and n_vis < 2:
            actions.append("schedule at least one additional clinic visit")
        if idx.get("anchor_complete_all3_last_anchor") == 0:
            actions.append("ensure UPDRS-III, MoCA and SCOPA-AUT are all administered at the next visit")
        if conf == 0 and supp != 1:
            actions.append("consider shorter inter-visit intervals to reduce latent state uncertainty")
        if actions:
            parts.append("Actionable steps: " + "; ".join(actions) + ".")

    display(HTML(f"""
    <div style="border-left: 4px solid #1565c0; padding: 12px 16px;
                background: #f5f7fa; border-radius: 2px; margin-bottom: 8px;">
        <div style="font-size: 12px; color: #333; line-height: 1.7;">
            {'<br><br>'.join(parts)}
        </div>
    </div>
    """))
    display(HTML(f"<p style='color:#999; font-size:10px;'>Evidence: {' + '.join(evidence)}</p>"))


def _insight_model_trustworthiness(fold: str, pid: int) -> None:
    idx_res = STORE.get_patient_indices(fold, pid)
    idx = idx_res.data
    demo = _get_demographics(pid)
    sex = _sex_label(demo.get("sex"))
    subj, obj, poss = _pronoun(sex)
    evidence: List[str] = [str(idx_res.pointer)]

    if idx is None:
        print(f"Patient {pid} not found.")
        return

    parts = []

    # Patient-level confidence
    conf = idx.get("confidence_flag")
    delta_iqr = idx.get("delta_latent_iqr_last_anchor")
    tau = idx.get("tau_conf_last_anchor")

    if isinstance(delta_iqr, float) and isinstance(tau, float):
        ratio = delta_iqr / tau if tau > 0 else float("inf")
        if ratio <= 0.5:
            conf_desc = f"well within the confidence threshold (ratio {ratio:.2f})"
        elif ratio <= 1.0:
            conf_desc = f"within the confidence threshold (ratio {ratio:.2f})"
        else:
            conf_desc = f"above the confidence threshold (ratio {ratio:.2f})"
        parts.append(
            f"For patient {pid}, the model's latent state uncertainty at the last anchor "
            f"is {delta_iqr:.4f}, {conf_desc} of {tau:.2f}. "
            + ("This means the model is confident in its estimate of {poss} current disease state."
               if ratio <= 1.0 else
               f"This means additional data would help the model better locate {poss} disease state.")
        )

    # Fold-level convergence
    try:
        conv_res = STORE.get_convergence(fold)
        evidence.append(str(conv_res.pointer))
        gs = conv_res.data.get("global_summary", {})
        met = gs.get("convergence_met", False)
        rhat = gs.get("max_r_hat")
        ess = gs.get("min_ess_bulk")

        if met:
            parts.append(
                f"The underlying Bayesian model for this cohort converged successfully "
                f"(max R-hat = {rhat}, min ESS = {ess}, 0 divergent transitions). "
                f"The global parameter estimates are statistically sound."
            )
        else:
            parts.append(
                f"Note: convergence criteria were not fully met for this cohort "
                f"(max R-hat = {rhat}). Results should be interpreted with extra caution."
            )
    except Exception:
        pass

    # Calibration for this patient's relevant events
    try:
        cal_res = STORE.get_threshold_calibration(fold)
        evidence.append(str(cal_res.pointer))
        cal = cal_res.data
        if not cal.empty:
            mean_ece = cal["expected_calibration_error"].mean()
            mean_bss = cal["brier_skill_score"].mean()

            if mean_ece < 0.03:
                cal_quality = "excellent calibration"
            elif mean_ece < 0.06:
                cal_quality = "good calibration"
            else:
                cal_quality = "moderate calibration with some room for improvement"

            parts.append(
                f"Threshold event calibration shows {cal_quality} "
                f"(mean ECE = {mean_ece:.4f} across all events). "
                f"Brier skill score averages {mean_bss:.4f} "
                + ("(positive, outperforming the base-rate benchmark)."
                   if mean_bss > 0 else "(near or below the base-rate benchmark).")
            )
    except Exception:
        pass

    # PPC flags
    try:
        ppc_res = STORE.get_ppc_summary(fold)
        evidence.append(str(ppc_res.pointer))
        ppc = ppc_res.data
        pit = ppc.get("pit_calibration", {})
        flagged_endpoints = [ep for ep, v in pit.items() if v.get("flag_ks_gt_0p10", 0) == 1]
        residual = ppc.get("residual_cross_endpoint_dependence", {})
        res_flagged = residual.get("flagged", 0) == 1

        if not flagged_endpoints and not res_flagged:
            parts.append(
                "Posterior predictive checks passed for all endpoints with no flagged "
                "calibration issues and minimal residual cross-endpoint dependence."
            )
        else:
            issues = []
            if flagged_endpoints:
                issues.append(f"PIT calibration flagged for: {', '.join(flagged_endpoints)}")
            if res_flagged:
                issues.append("residual cross-endpoint dependence above threshold")
            parts.append(
                f"Posterior predictive checks identified: {'; '.join(issues)}. "
                f"This does not invalidate results but suggests the model may be slightly "
                f"miscalibrated for these specific endpoints."
            )
    except Exception:
        pass

    display(HTML(f"""
    <div style="border-left: 4px solid #666666; padding: 12px 16px;
                background: #f5f7fa; border-radius: 2px; margin-bottom: 8px;">
        <div style="font-size: 12px; color: #333; line-height: 1.7;">
            {'<br><br>'.join(parts)}
        </div>
    </div>
    """))
    display(HTML(f"<p style='color:#999; font-size:10px;'>Evidence: {' + '.join(evidence)}</p>"))


# ============================================================================
# Insight dispatcher
# ============================================================================
_INSIGHT_HANDLERS = {
    "comprehensive_profile": _insight_comprehensive_profile,
    "motor_outlook": _insight_motor_outlook,
    "cognitive_vulnerability": _insight_cognitive_vulnerability,
    "multi_domain_landscape": _insight_multi_domain,
    "cohort_benchmarking": _insight_cohort_benchmarking,
    "trajectory_dynamics": _insight_trajectory_dynamics,
    "data_completeness": _insight_data_completeness,
    "model_trustworthiness": _insight_model_trustworthiness,
}

# ============================================================================
# Widgets
# ============================================================================
_r_fold = widgets.Dropdown(
    options=list(_FOLD_LABELS.values()),
    value=list(_FOLD_LABELS.values())[0],
    description="Validation cohort:",
    layout=widgets.Layout(width="260px"),
    style={"description_width": "auto"},
)

_r_sex_filter = widgets.Dropdown(
    options=["All patients", "Female only", "Male only"],
    value="All patients",
    description="Sex:",
    layout=widgets.Layout(width="180px"),
    style={"description_width": "auto"},
)

_r_patient = widgets.Dropdown(
    options=_first_pids,
    value=_first_pids[0] if _first_pids else None,
    description="Patient ID:",
    layout=widgets.Layout(width="230px"),
    style={"description_width": "auto"},
)

_r_question = widgets.Text(
    value="",
    placeholder="e.g. What is the risk of motor worsening at 1 year?",
    description="",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "0px"},
)

_r_ask_btn = widgets.Button(
    description="Ask",
    button_style="primary",
    layout=widgets.Layout(width="80px"),
)

_r_clear_btn = widgets.Button(
    description="Clear",
    button_style="",
    layout=widgets.Layout(width="80px"),
)

_r_out = widgets.Output()
_r_history: List[Tuple[str, str, int]] = []

# ============================================================================
# Patient list filtering
# ============================================================================
def _r_filtered_patients(fold_id: str, sex_filter: str) -> List[int]:
    all_pids = STORE.patient_ids_for_fold(fold_id)
    if sex_filter == "All patients":
        return all_pids
    sex_map = _sex_by_fold.get(fold_id, {})
    target = "F" if sex_filter == "Female only" else "M"
    return [p for p in all_pids if sex_map.get(p, "?") == target]

def _r_refresh_patients() -> None:
    fold_id = _FOLD_REVERSE.get(_r_fold.value, STORE.folds[0])
    pids = _r_filtered_patients(fold_id, _r_sex_filter.value)
    _r_patient.options = pids
    if pids:
        _r_patient.value = pids[0]

_r_fold.observe(lambda c: _r_refresh_patients(), names="value")
_r_sex_filter.observe(lambda c: _r_refresh_patients(), names="value")

# ============================================================================
# Insight card rendering
# ============================================================================
def _build_card_buttons() -> widgets.VBox:
    rows = []
    for card in _INSIGHT_CARDS:
        btn = widgets.Button(
            description="",
            layout=widgets.Layout(width="100%", height="0px", padding="0px",
                                  margin="0px", border="none"),
        )
        # Use an Output for each card to render HTML
        card_html = widgets.HTML(value=f"""
        <div style="background: white; border: 1px solid #e0e0e0; border-left: 4px solid {card.icon_color};
                    padding: 10px 14px; border-radius: 3px; cursor: pointer;
                    transition: box-shadow 0.2s;"
             onmouseover="this.style.boxShadow='0 2px 8px rgba(0,0,0,0.1)'"
             onmouseout="this.style.boxShadow='none'"
             title="Click to generate this insight">
            <strong style="font-size: 12px; color: {card.icon_color};">{card.title}</strong><br>
            <span style="font-size: 11px; color: #777;">{card.subtitle}</span>
        </div>
        """)

        click_btn = widgets.Button(
            description=f"  {card.title}",
            layout=widgets.Layout(width="100%", height="auto", min_height="52px"),
            style={"button_color": "white", "font_weight": "normal"},
            tooltip=card.subtitle,
        )

        key = card.key
        color = card.icon_color

        def make_click_handler(k: str, c: str):
            def handler(_b: widgets.Button) -> None:
                _r_out.clear_output(wait=True)
                with _r_out:
                    fold = _FOLD_REVERSE.get(_r_fold.value, STORE.folds[0])
                    pid = _r_patient.value
                    if pid is None:
                        print("No patient selected.")
                        return
                    demo = _get_demographics(pid)
                    sex = _sex_label(demo.get("sex"))
                    age = demo.get("age_baseline_years")
                    cohort = demo.get("cohort", "")
                    age_str = f"age {age:.0f}" if isinstance(age, (int, float)) else ""
                    ctx = [p for p in [sex, age_str, cohort] if p]

                    display(HTML(f"""
                    <div style="background: #f5f5f5; border-left: 4px solid {c};
                                padding: 8px 14px; margin-bottom: 10px; border-radius: 2px;">
                        <strong style="font-size: 13px;">Patient {pid}</strong>
                        {(' (' + ', '.join(ctx) + ')') if ctx else ''}
                    </div>
                    """))

                    try:
                        handler_fn = _INSIGHT_HANDLERS.get(k)
                        if handler_fn:
                            handler_fn(fold, pid)
                    except Exception as e:
                        print(f"Error: {e}")
                        import traceback
                        traceback.print_exc()

                    _r_history.append((k, fold, pid))
                    if len(_r_history) > 20:
                        _r_history.pop(0)
            return handler
        click_btn.on_click(make_click_handler(key, color))
        rows.append(click_btn)

    # Arrange in 2-column grid
    grid_rows = []
    for i in range(0, len(rows), 2):
        pair = rows[i:i+2]
        if len(pair) == 2:
            grid_rows.append(widgets.HBox(pair, layout=widgets.Layout(gap="6px", width="100%")))
        else:
            grid_rows.append(widgets.HBox(pair, layout=widgets.Layout(width="100%")))

    return widgets.VBox(grid_rows, layout=widgets.Layout(gap="4px"))


_card_grid = _build_card_buttons()

# ============================================================================
# Free-text query handler
# ============================================================================
def _r_run_query(_btn: Any) -> None:
    _r_out.clear_output(wait=True)
    with _r_out:
        fold = _FOLD_REVERSE.get(_r_fold.value, STORE.folds[0])
        pid = _r_patient.value
        q = _r_question.value.strip()

        if not q:
            print("Enter a question or click an insight card above.")
            return
        if pid is None:
            print("No patient selected.")
            return

        demo = _get_demographics(pid)
        sex = _sex_label(demo.get("sex"))
        age = demo.get("age_baseline_years")
        cohort = demo.get("cohort", "")
        age_str = f"age {age:.0f}" if isinstance(age, (int, float)) else ""
        ctx = [p for p in [sex, age_str, cohort] if p]

        display(HTML(f"""
        <div style="background: #f5f5f5; border-left: 4px solid #0f3460;
                    padding: 8px 14px; margin-bottom: 10px; border-radius: 2px;">
            <strong style="font-size: 13px;">Patient {pid}</strong>
            {(' (' + ', '.join(ctx) + ')') if ctx else ''}
            &nbsp;&nbsp;|&nbsp;&nbsp;
            <em style="color: #555; font-size: 12px;">{q}</em>
        </div>
        """))

        try:
            answer = REASONER.ask(q, fold, pid)
            answer.display()
        except Exception as e:
            print(f"Error: {e}")
            import traceback
            traceback.print_exc()

_r_ask_btn.on_click(_r_run_query)
_r_question.on_submit(lambda _w: _r_run_query(None))

def _r_clear(_btn: Any) -> None:
    _r_out.clear_output()
    _r_question.value = ""
_r_clear_btn.on_click(_r_clear)

# ============================================================================
# Layout
# ============================================================================
_r_controls = widgets.HBox(
    [_r_fold, _r_sex_filter, _r_patient],
    layout=widgets.Layout(gap="6px"),
)

_r_cards_label = widgets.HTML(
    value="<p style='font-size: 11px; color: #888; margin: 8px 0 4px 0;'>Clinical insight cards:</p>"
)

_r_freetext_label = widgets.HTML(
    value="<p style='font-size: 11px; color: #888; margin: 10px 0 2px 0;'>Or ask a free-text question:</p>"
)

_r_input_row = widgets.HBox(
    [_r_question, _r_ask_btn, _r_clear_btn],
    layout=widgets.Layout(gap="4px", width="100%"),
)

_r_ui = widgets.VBox([
    _r_controls,
    _r_cards_label,
    _card_grid,
    _r_freetext_label,
    _r_input_row,
    _r_out,
], layout=widgets.Layout(gap="4px"))



In [ ]:
%%capture _dd_code_log
# @title D4. Deep-dive code definitions 

import textwrap as _tw

import base64 as _b64
__deep_dive_a_code = _b64.b64decode('...').decode()
def _deep_dive_a():
    exec(__deep_dive_a_code, globals())

import base64 as _b64
__deep_dive_b_code = _b64.b64decode('...').decode()
def _deep_dive_b():
    exec(__deep_dive_b_code, globals())

import base64 as _b64
__deep_dive_c_code = _b64.b64decode('...').decode()
def _deep_dive_c():
    exec(__deep_dive_c_code, globals())

import base64 as _b64
__worked_examples_code = _b64.b64decode('IyAtLS0gV29ya2VkIGV4YW1wbGVzOiBncm91bmRlZCBjbGluaWNhbCBxdWVzdGlvbnMgLS0tCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgU2VsZWN0IHJlcHJlc2VudGF0aXZlIHBhdGllbnRzCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIF9waWNrX3JlcHJlc2VudGF0aXZlX3BhdGllbnRzKCkgLT4gTGlzdFtEaWN0W3N0ciwgQW55XV06CiAgICAiIiIKICAgIFNlbGVjdCBwYXRpZW50cyB0aGF0IGlsbHVzdHJhdGUgZGlmZmVyZW50IGNsaW5pY2FsIHNjZW5hcmlvczoKICAgIHByZWRpY3Rpb25zIGF2YWlsYWJsZSwgYWRkaXRpb25hbCBkYXRhIG5lZWRlZCwgZGlmZmVyZW50IGNvaG9ydHMsCiAgICBoaWdoL2xvdyBidXJkZW4sIG1hbGUvZmVtYWxlLgogICAgIiIiCiAgICBwaWNrcyA9IFtdCiAgICB1c2VkX3BpZHMgPSBzZXQoKQoKICAgIGZvciBmb2xkIGluIFNUT1JFLmZvbGRzOgogICAgICAgIGlmIGZvbGQgbm90IGluIGFsbF9pbmRpY2VzX2RmczoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZHggPSBhbGxfaW5kaWNlc19kZnNbZm9sZF0KICAgICAgICBpZiBpZHguZW1wdHk6CiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIGhhc19jb25mID0gImNvbmZpZGVuY2VfZmxhZyIgaW4gaWR4LmNvbHVtbnMKICAgICAgICBoYXNfc3VwcCA9ICJzdXBwcmVzc2VkX2F0X2xhc3RfYW5jaG9yIiBpbiBpZHguY29sdW1ucwogICAgICAgIGhhc19tdiA9ICJtb3Rvcl92b2xhdGlsaXR5X2luZGV4IiBpbiBpZHguY29sdW1ucwogICAgICAgIGhhc19jZiA9ICJuZWFyX3Rlcm1fY29nbml0aXZlX2ZyYWdpbGl0eV9pbmRleCIgaW4gaWR4LmNvbHVtbnMKCiAgICAgICAgaWYgaGFzX2NvbmYgYW5kIGhhc19zdXBwOgogICAgICAgICAgICAjIFBhdGllbnQgd2l0aCBwcmVkaWN0aW9ucyBhdmFpbGFibGUKICAgICAgICAgICAgcmVhZHkgPSBpZHhbKGlkeFsiY29uZmlkZW5jZV9mbGFnIl0gPT0gMSkgJiAoaWR4WyJzdXBwcmVzc2VkX2F0X2xhc3RfYW5jaG9yIl0gPT0gMCldCiAgICAgICAgICAgIGlmIG5vdCByZWFkeS5lbXB0eSBhbmQgaGFzX212IGFuZCBoYXNfY2Y6CiAgICAgICAgICAgICAgICAjIEhpZ2ggbW90b3Igdm9sYXRpbGl0eQogICAgICAgICAgICAgICAgcmVhZHlfbXYgPSByZWFkeS5kcm9wbmEoc3Vic2V0PVsibW90b3Jfdm9sYXRpbGl0eV9pbmRleCJdKQogICAgICAgICAgICAgICAgaWYgbm90IHJlYWR5X212LmVtcHR5OgogICAgICAgICAgICAgICAgICAgIGhpZ2hfbXYgPSByZWFkeV9tdi5sb2NbcmVhZHlfbXZbIm1vdG9yX3ZvbGF0aWxpdHlfaW5kZXgiXS5pZHhtYXgoKV0KICAgICAgICAgICAgICAgICAgICBwaWQgPSBpbnQoaGlnaF9tdlsicGF0aWVudF9pZCJdKQogICAgICAgICAgICAgICAgICAgIGlmIHBpZCBub3QgaW4gdXNlZF9waWRzOgogICAgICAgICAgICAgICAgICAgICAgICBwaWNrcy5hcHBlbmQoeyJwaWQiOiBwaWQsICJmb2xkIjogZm9sZCwgInRhZyI6ICJoaWdoX3ZvbGF0aWxpdHkifSkKICAgICAgICAgICAgICAgICAgICAgICAgdXNlZF9waWRzLmFkZChwaWQpCgogICAgICAgICAgICAgICAgIyBIaWdoIGNvZ25pdGl2ZSBmcmFnaWxpdHkKICAgICAgICAgICAgICAgIHJlYWR5X2NmID0gcmVhZHkuZHJvcG5hKHN1YnNldD1bIm5lYXJfdGVybV9jb2duaXRpdmVfZnJhZ2lsaXR5X2luZGV4Il0pCiAgICAgICAgICAgICAgICBpZiBub3QgcmVhZHlfY2YuZW1wdHk6CiAgICAgICAgICAgICAgICAgICAgaGlnaF9jZiA9IHJlYWR5X2NmLmxvY1tyZWFkeV9jZlsibmVhcl90ZXJtX2NvZ25pdGl2ZV9mcmFnaWxpdHlfaW5kZXgiXS5pZHhtYXgoKV0KICAgICAgICAgICAgICAgICAgICBwaWQgPSBpbnQoaGlnaF9jZlsicGF0aWVudF9pZCJdKQogICAgICAgICAgICAgICAgICAgIGlmIHBpZCBub3QgaW4gdXNlZF9waWRzOgogICAgICAgICAgICAgICAgICAgICAgICBwaWNrcy5hcHBlbmQoeyJwaWQiOiBwaWQsICJmb2xkIjogZm9sZCwgInRhZyI6ICJoaWdoX2ZyYWdpbGl0eSJ9KQogICAgICAgICAgICAgICAgICAgICAgICB1c2VkX3BpZHMuYWRkKHBpZCkKCiAgICAgICAgICAgICAgICAjIFN0YWJsZSBsb3ctYnVyZGVuIHBhdGllbnQKICAgICAgICAgICAgICAgIGlmICJ4TV9sYXN0IiBpbiByZWFkeS5jb2x1bW5zOgogICAgICAgICAgICAgICAgICAgIGxvd19idXJkZW4gPSByZWFkeS5kcm9wbmEoc3Vic2V0PVsieE1fbGFzdCJdKQogICAgICAgICAgICAgICAgICAgIGlmIG5vdCBsb3dfYnVyZGVuLmVtcHR5OgogICAgICAgICAgICAgICAgICAgICAgICBsYiA9IGxvd19idXJkZW4ubG9jW2xvd19idXJkZW5bInhNX2xhc3QiXS5pZHhtaW4oKV0KICAgICAgICAgICAgICAgICAgICAgICAgcGlkID0gaW50KGxiWyJwYXRpZW50X2lkIl0pCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIHBpZCBub3QgaW4gdXNlZF9waWRzOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgcGlja3MuYXBwZW5kKHsicGlkIjogcGlkLCAiZm9sZCI6IGZvbGQsICJ0YWciOiAibG93X2J1cmRlbiJ9KQogICAgICAgICAgICAgICAgICAgICAgICAgICAgdXNlZF9waWRzLmFkZChwaWQpCgogICAgICAgICAgICAjIFBhdGllbnQgbmVlZGluZyBhZGRpdGlvbmFsIGRhdGEKICAgICAgICAgICAgaWYgaGFzX3N1cHA6CiAgICAgICAgICAgICAgICBuZWVkc19kYXRhID0gaWR4W2lkeFsic3VwcHJlc3NlZF9hdF9sYXN0X2FuY2hvciJdID09IDFdCiAgICAgICAgICAgICAgICBpZiBub3QgbmVlZHNfZGF0YS5lbXB0eToKICAgICAgICAgICAgICAgICAgICBuZCA9IG5lZWRzX2RhdGEuaWxvY1swXQogICAgICAgICAgICAgICAgICAgIHBpZCA9IGludChuZFsicGF0aWVudF9pZCJdKQogICAgICAgICAgICAgICAgICAgIGlmIHBpZCBub3QgaW4gdXNlZF9waWRzOgogICAgICAgICAgICAgICAgICAgICAgICBwaWNrcy5hcHBlbmQoeyJwaWQiOiBwaWQsICJmb2xkIjogZm9sZCwgInRhZyI6ICJuZWVkc19kYXRhIn0pCiAgICAgICAgICAgICAgICAgICAgICAgIHVzZWRfcGlkcy5hZGQocGlkKQoKICAgICAgICBpZiBsZW4ocGlja3MpID49IDU6CiAgICAgICAgICAgIGJyZWFrCgogICAgIyBJZiB3ZSBzdGlsbCBuZWVkIG1vcmUsIGdyYWIgZnJvbSByZW1haW5pbmcgZm9sZHMKICAgIGZvciBmb2xkIGluIFNUT1JFLmZvbGRzOgogICAgICAgIGlmIGxlbihwaWNrcykgPj0gNToKICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBmb2xkIG5vdCBpbiBhbGxfaW5kaWNlc19kZnM6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWR4ID0gYWxsX2luZGljZXNfZGZzW2ZvbGRdCiAgICAgICAgZm9yIF8sIHJvdyBpbiBpZHguaXRlcnJvd3MoKToKICAgICAgICAgICAgcGlkID0gaW50KHJvd1sicGF0aWVudF9pZCJdKQogICAgICAgICAgICBpZiBwaWQgbm90IGluIHVzZWRfcGlkczoKICAgICAgICAgICAgICAgIHBpY2tzLmFwcGVuZCh7InBpZCI6IHBpZCwgImZvbGQiOiBmb2xkLCAidGFnIjogImdlbmVyYWwifSkKICAgICAgICAgICAgICAgIHVzZWRfcGlkcy5hZGQocGlkKQogICAgICAgICAgICAgICAgYnJlYWsKCiAgICByZXR1cm4gcGlja3NbOjVdCgoKX2V4YW1wbGVfcGF0aWVudHMgPSBfcGlja19yZXByZXNlbnRhdGl2ZV9wYXRpZW50cygpCgppZiBub3QgX2V4YW1wbGVfcGF0aWVudHM6CiAgICBwcmludCgiQ291bGQgbm90IHNlbGVjdCByZXByZXNlbnRhdGl2ZSBwYXRpZW50cy4gVXNpbmcgZmlyc3QgYXZhaWxhYmxlLiIpCiAgICBmb2xkMCA9IFNUT1JFLmZvbGRzWzBdCiAgICBwaWRzMCA9IFNUT1JFLnBhdGllbnRfaWRzX2Zvcl9mb2xkKGZvbGQwKQogICAgaWYgcGlkczA6CiAgICAgICAgX2V4YW1wbGVfcGF0aWVudHMgPSBbeyJwaWQiOiBwaWRzMFswXSwgImZvbGQiOiBmb2xkMCwgInRhZyI6ICJmYWxsYmFjayJ9XQoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgRXhhbXBsZSBxdWVzdGlvbnMKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpfV09SS0VEX0VYQU1QTEVTID0gWwogICAgewogICAgICAgICJwYXRpZW50X2lkeCI6IDAsCiAgICAgICAgInF1ZXN0aW9uIjogIlRlbGwgbWUgYWJvdXQgdGhpcyBwYXRpZW50IiwKICAgICAgICAiaW50ZW50IjogIlBhdGllbnQgc3VtbWFyeSB3aXRoIGRlbW9ncmFwaGljcywgZGlzZWFzZSBzdGF0ZSBhbmQgcHJlZGljdGlvbiByZWFkaW5lc3MiLAogICAgfSwKICAgIHsKICAgICAgICAicGF0aWVudF9pZHgiOiAwLAogICAgICAgICJxdWVzdGlvbiI6ICJXaGF0IGlzIHRoZSByaXNrIG9mIG1vdG9yIHdvcnNlbmluZyBvZiAxMCBVUERSUyBwb2ludHMgYXQgMSB5ZWFyPyIsCiAgICAgICAgImludGVudCI6ICJIb3Jpem9uLXNwZWNpZmljIHJpc2sgcXVlcnkgd2l0aCBjbGluaWNhbCBldmVudCBtYXRjaGluZyIsCiAgICB9LAogICAgewogICAgICAgICJwYXRpZW50X2lkeCI6IDEgaWYgbGVuKF9leGFtcGxlX3BhdGllbnRzKSA+IDEgZWxzZSAwLAogICAgICAgICJxdWVzdGlvbiI6ICJXaGF0IGlzIHRoZSBjb2duaXRpdmUgZnJhZ2lsaXR5IGZvciB0aGlzIHBhdGllbnQ/IiwKICAgICAgICAiaW50ZW50IjogIkluZGV4IGxvb2t1cCB3aXRoIGNvaG9ydCBwZXJjZW50aWxlIGNvbnRleHQgYW5kIHJpc2sgbGV2ZWwiLAogICAgfSwKICAgIHsKICAgICAgICAicGF0aWVudF9pZHgiOiAxIGlmIGxlbihfZXhhbXBsZV9wYXRpZW50cykgPiAxIGVsc2UgMCwKICAgICAgICAicXVlc3Rpb24iOiAiSG93IGRvZXMgdGhpcyBwYXRpZW50IGNvbXBhcmUgdG8gdGhlIHJlc3Qgb2YgdGhlIGNvaG9ydD8iLAogICAgICAgICJpbnRlbnQiOiAiQ3Jvc3MtaW5kZXggY29ob3J0IGJlbmNobWFya2luZyB3aXRoIHBlcmNlbnRpbGUgcmFua3MiLAogICAgfSwKICAgIHsKICAgICAgICAicGF0aWVudF9pZHgiOiAyIGlmIGxlbihfZXhhbXBsZV9wYXRpZW50cykgPiAyIGVsc2UgMCwKICAgICAgICAicXVlc3Rpb24iOiAiSXMgdGhlIGRpc2Vhc2UgdHJhamVjdG9yeSBhY2NlbGVyYXRpbmcgZm9yIHRoaXMgcGF0aWVudD8iLAogICAgICAgICJpbnRlbnQiOiAiQWNjZWxlcmF0aW9uIGZsYWcgd2l0aCBjb2hvcnQgcHJldmFsZW5jZSBjb250ZXh0IiwKICAgIH0sCiAgICB7CiAgICAgICAgInBhdGllbnRfaWR4IjogMCwKICAgICAgICAicXVlc3Rpb24iOiAiV2hhdCBpcyB0aGUgbW90b3Igdm9sYXRpbGl0eSBmb3IgdGhpcyBwYXRpZW50PyIsCiAgICAgICAgImludGVudCI6ICJNb3RvciB2b2xhdGlsaXR5IGluZGV4IHdpdGggc3ByZWFkIGludGVycHJldGF0aW9uIiwKICAgIH0sCiAgICB7CiAgICAgICAgInBhdGllbnRfaWR4IjogMyBpZiBsZW4oX2V4YW1wbGVfcGF0aWVudHMpID4gMyBlbHNlIDAsCiAgICAgICAgInF1ZXN0aW9uIjogIkFyZSBwcmVkaWN0aW9ucyBhdmFpbGFibGUgZm9yIHRoaXMgcGF0aWVudCBhbmQgd2hhdCB3b3VsZCBpbXByb3ZlIHRoZW0/IiwKICAgICAgICAiaW50ZW50IjogIlByZWRpY3Rpb24gcmVhZGluZXNzIHdpdGggY29uc3RydWN0aXZlIGRhdGEgcmVxdWlyZW1lbnRzIiwKICAgIH0sCiAgICB7CiAgICAgICAgInBhdGllbnRfaWR4IjogMCwKICAgICAgICAicXVlc3Rpb24iOiAiSG93IGFjY3VyYXRlIGlzIHRoZSBtb2RlbCBmb3IgdGhpcyB2YWxpZGF0aW9uIGNvaG9ydD8iLAogICAgICAgICJpbnRlbnQiOiAiQWdncmVnYXRlIG1ldHJpY3Mgd2l0aCBjbGluaWNhbCBzY2FsZSBjb250ZXh0IiwKICAgIH0sCiAgICB7CiAgICAgICAgInBhdGllbnRfaWR4IjogMCwKICAgICAgICAicXVlc3Rpb24iOiAiSXMgdGhlIG1vZGVsIHdlbGwgY2FsaWJyYXRlZCBmb3IgdGhyZXNob2xkIGV2ZW50cz8iLAogICAgICAgICJpbnRlbnQiOiAiQ2FsaWJyYXRpb24gcXVhbGl0eSBhc3Nlc3NtZW50IHdpdGggRUNFIGFuZCBCcmllciBza2lsbCIsCiAgICB9LAogICAgewogICAgICAgICJwYXRpZW50X2lkeCI6IDAsCiAgICAgICAgInF1ZXN0aW9uIjogIldvdWxkIGEgcmV2aWV3ZXIgYWNjZXB0IHRoZSBpbmZlcmVuY2UgZGlhZ25vc3RpY3MgZm9yIHRoaXMgY29ob3J0PyIsCiAgICAgICAgImludGVudCI6ICJNdWx0aS1kaWFnbm9zdGljIHN5bnRoZXNpczogY29udmVyZ2VuY2UsIFBQQywgY2FsaWJyYXRpb24iLAogICAgfSwKXQoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgUnVuIGV4YW1wbGVzCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpmb3IgaSwgZXhhbXBsZSBpbiBlbnVtZXJhdGUoX1dPUktFRF9FWEFNUExFUywgMSk6CiAgICBwaWR4ID0gZXhhbXBsZVsicGF0aWVudF9pZHgiXQogICAgaWYgcGlkeCA+PSBsZW4oX2V4YW1wbGVfcGF0aWVudHMpOgogICAgICAgIHBpZHggPSAwCiAgICBwYXRpZW50X2luZm8gPSBfZXhhbXBsZV9wYXRpZW50c1twaWR4XQogICAgcGlkID0gcGF0aWVudF9pbmZvWyJwaWQiXQogICAgZm9sZCA9IHBhdGllbnRfaW5mb1siZm9sZCJdCiAgICBxdWVzdGlvbiA9IGV4YW1wbGVbInF1ZXN0aW9uIl0KICAgIGludGVudF9kZXNjID0gZXhhbXBsZVsiaW50ZW50Il0KCiAgICAjIFBhdGllbnQgY29udGV4dAogICAgZGVtbyA9IF9nZXRfZGVtb2dyYXBoaWNzKHBpZCkKICAgIHNleCA9IF9zZXhfbGFiZWwoZGVtby5nZXQoInNleCIpKQogICAgYWdlID0gZGVtby5nZXQoImFnZV9iYXNlbGluZV95ZWFycyIpCiAgICBjb2hvcnQgPSBkZW1vLmdldCgiY29ob3J0IiwgIiIpCiAgICBhZ2Vfc3RyID0gZiJhZ2Uge2FnZTouMGZ9IiBpZiBpc2luc3RhbmNlKGFnZSwgKGludCwgZmxvYXQpKSBlbHNlICIiCiAgICBjdHggPSBbcCBmb3IgcCBpbiBbc2V4LCBhZ2Vfc3RyLCBjb2hvcnRdIGlmIHBdCiAgICBjdHhfc3RyID0gZiIgKHsnLCAnLmpvaW4oY3R4KX0pIiBpZiBjdHggZWxzZSAiIgoKICAgICMgU2VjdGlvbiBoZWFkZXIKICAgIGRpc3BsYXkoSFRNTChmIiIiCiAgICA8ZGl2IHN0eWxlPSJiYWNrZ3JvdW5kOiAjZjBmMmY1OyBib3JkZXItbGVmdDogNHB4IHNvbGlkICMwZjM0NjA7CiAgICAgICAgICAgICAgICBwYWRkaW5nOiAxMHB4IDE0cHg7IG1hcmdpbi10b3A6IDE2cHg7IG1hcmdpbi1ib3R0b206IDZweDsKICAgICAgICAgICAgICAgIGJvcmRlci1yYWRpdXM6IDJweDsiPgogICAgICAgIDxzdHJvbmcgc3R5bGU9ImZvbnQtc2l6ZTogMTNweDsgY29sb3I6ICMwZjM0NjA7Ij4KICAgICAgICAgICAgRXhhbXBsZSB7aX06IHtxdWVzdGlvbn0KICAgICAgICA8L3N0cm9uZz4KICAgICAgICA8ZGl2IHN0eWxlPSJmb250LXNpemU6IDExcHg7IGNvbG9yOiAjNzc3OyBtYXJnaW4tdG9wOiAycHg7Ij4KICAgICAgICAgICAgUGF0aWVudCB7cGlkfXtjdHhfc3RyfQogICAgICAgICAgICAmbmJzcDt8Jm5ic3A7IHtfRk9MRF9MQUJFTFMuZ2V0KGZvbGQsIGZvbGQpfQogICAgICAgICAgICAmbmJzcDt8Jm5ic3A7IDxlbT5JbnRlbnQ6IHtpbnRlbnRfZGVzY308L2VtPgogICAgICAgIDwvZGl2PgogICAgPC9kaXY+CiAgICAiIiIpKQoKICAgIHRyeToKICAgICAgICBhbnN3ZXIgPSBSRUFTT05FUi5hc2socXVlc3Rpb24sIGZvbGQsIHBpZCkKICAgICAgICBhbnN3ZXIuZGlzcGxheSgpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcHJpbnQoZiJFcnJvcjoge2V9IikKICAgICAgICBpbXBvcnQgdHJhY2ViYWNrCiAgICAgICAgdHJhY2ViYWNrLnByaW50X2V4YygpCgogICAgcHJpbnQoKQoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgU3VtbWFyeQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KZGlzcGxheShIVE1MKCIiIgo8ZGl2IHN0eWxlPSJiYWNrZ3JvdW5kOiAjZjlmOWY5OyBwYWRkaW5nOiAxMnB4IDE2cHg7IGJvcmRlci1yYWRpdXM6IDRweDsKICAgICAgICAgICAgbWFyZ2luLXRvcDogMTZweDsgZm9udC1zaXplOiAxMnB4OyBjb2xvcjogIzQ0NDsgbGluZS1oZWlnaHQ6IDEuNzsiPgogICAgPHN0cm9uZz5XaGF0IHRoZXNlIGV4YW1wbGVzIGRlbW9uc3RyYXRlOjwvc3Ryb25nPjxicj48YnI+CiAgICBUaGUgY2xpbmljYWwgcmVhc29uZXIgaGFuZGxlcyBkaXZlcnNlIHF1ZXN0aW9uIHR5cGVzIHdpdGhvdXQgYW55IGV4dGVybmFsIEFQSSBvcgogICAgbGFuZ3VhZ2UgbW9kZWwuIEV2ZXJ5IGFuc3dlciBpcyBjb21wdXRlZCBkZXRlcm1pbmlzdGljYWxseSBmcm9tIHRoZSBTWU1QSE9OWSBidW5kbGUKICAgIGZpbGVzIGFuZCBpbmNsdWRlcyBleHBsaWNpdCBldmlkZW5jZSBwb2ludGVycyB0cmFjaW5nIGVhY2ggY2xhaW0gdG8gaXRzIHNvdXJjZS48YnI+PGJyPgogICAgUXVlc3Rpb24gdHlwZXMgY292ZXJlZDogcGF0aWVudCBzdW1tYXJpZXMgd2l0aCBkZW1vZ3JhcGhpY3MsIGhvcml6b24tc3BlY2lmaWMgcmlzawogICAgcXVlcmllcywgZGVyaXZlZCBjbGluaWNhbCBpbmRpY2VzIChjb2duaXRpdmUgZnJhZ2lsaXR5LCBtb3RvciB2b2xhdGlsaXR5LCB0cmFqZWN0b3J5CiAgICBhY2NlbGVyYXRpb24pLCBjb2hvcnQgYmVuY2htYXJraW5nIHdpdGggcGVyY2VudGlsZSByYW5rcywgcHJlZGljdGlvbiByZWFkaW5lc3MKICAgIGFzc2Vzc21lbnQgd2l0aCBjb25zdHJ1Y3RpdmUgZ3VpZGFuY2UsIGFnZ3JlZ2F0ZSBtb2RlbCBhY2N1cmFjeSwgdGhyZXNob2xkIGV2ZW50CiAgICBjYWxpYnJhdGlvbiwgYW5kIG11bHRpLWRpYWdub3N0aWMgcmV2aWV3ZXIgc3ludGhlc2lzLjxicj48YnI+CiAgICBXaGVuIHByZWRpY3Rpb25zIGFyZSBub3QgYXZhaWxhYmxlIGZvciBhIHBhdGllbnQsIHRoZSByZWFzb25lciBleHBsYWlucyB3aGF0CiAgICBzcGVjaWZpYyBjbGluaWNhbCBkYXRhIHdvdWxkIGJlIG5lZWRlZCB0byBnZW5lcmF0ZSB0aGVtLCByYXRoZXIgdGhhbiBzaW1wbHkKICAgIHJlcG9ydGluZyBtaXNzaW5nIHZhbHVlcy4gQWxsIGNvaG9ydCBjb21wYXJpc29ucyB1c2UgcGVyY2VudGlsZSByYW5rcyBjb21wdXRlZAogICAgZnJvbSB0aGUgc2FtZSB2YWxpZGF0aW9uIGNvaG9ydCB0byBlbnN1cmUgbGlrZS1mb3ItbGlrZSBjb250ZXh0Lgo8L2Rpdj4KIiIiKSkKCnByaW50KCJXb3JrZWQgZXhhbXBsZXMgY29tcGxldGUuIik=').decode()
def _worked_examples():
    exec(__worked_examples_code, globals())



In [ ]:
# E. SYMPHONY explorer

_main_out = widgets.Output()
_run_btn = widgets.Button(
    description="Run SYMPHONY explorer",
    button_style="primary",
    layout=widgets.Layout(width="280px", height="38px"),
    style={"font_weight": "bold"},
)
_progress_html = widgets.HTML(value="")

def _run_workflow(_btn):
    _run_btn.disabled = True
    _progress_html.value = '<span style="font-size:12px;color:#555;">Loading...</span>'

    with _main_out:
        clear_output(wait=True)

        # Bundle summary
        _meta = STORE.get_metadata().data
        _ds = _meta.get("data_source", {})
        display(HTML(f"""
        <div style="background:#f5f7fa;border-left:4px solid #0f3460;
                    padding:10px 16px;margin-bottom:10px;border-radius:2px;">
            <strong style="font-size:13px;">SYMPHONY bundle loaded</strong>
            <span style="color:#555;font-size:12px;margin-left:12px;">
                Run: {_meta.get("run_id","?")} |
                Source: {_ds.get("name","?")} |
                N={_ds.get("n_participants","?")} |
                Visits: {_ds.get("n_visits","?")} |
                Folds: {len(STORE.folds)}
            </span>
        </div>
        """))

        # Convergence one-liner
        _cv = STORE.convergence_across_folds().data
        _cv_ok = _cv["convergence_met"].astype(bool).all()
        _cv_c = "#4caf50" if _cv_ok else "#e53935"
        _cv_t = f"all {len(STORE.folds)} folds passed" if _cv_ok else "issues detected"
        display(HTML(f"""
        <p style="font-size:12px;color:#444;margin:0 0 12px 0;">
            Convergence: <span style="color:{_cv_c};font-weight:600;">{_cv_t}</span>
        </p>
        """))

        # Expandable validation
        display(HTML("""
        <details style="margin-bottom:14px;">
            <summary style="font-size:11px;color:#888;cursor:pointer;">
                Bundle validation details</summary>
            <div style="padding:6px 0;font-size:11px;color:#666;">
                All required files present. Per-fold validation passed.
                Expand the hidden cells above for the full manifest.
            </div>
        </details>
        """))

        # Clinical explorer
        display(HTML("""
        <div style="background:linear-gradient(135deg,#1a1a2e 0%,#16213e 50%,#0f3460 100%);
                    padding:18px 24px;border-radius:8px;margin-bottom:12px;">
            <h2 style="color:#e0e0e0;margin:0;font-family:'Segoe UI',sans-serif;
                       font-weight:500;letter-spacing:0.5px;">
                SYMPHONY clinical explorer
            </h2>
            <p style="color:#9a9abf;margin:4px 0 0 0;font-size:13px;">
                Patient-level exploration grounded in model outputs.
                Every value traces to a source file.
            </p>
        </div>
        """))

        _row1 = widgets.HBox([_w_fold, _w_sex_filter, _w_patient])
        _row2 = widgets.HBox([_w_event, _w_endpoint])
        _expl = widgets.VBox([_row1, _row2, _w_tab, _w_out],
                              layout=widgets.Layout(gap="6px"))
        display(_expl)
        _on_update()

        display(HTML('<hr style="border:none;border-top:1px solid #ddd;margin:20px 0;">'))

        # Clinical reasoner
        display(HTML("""
        <div style="background:linear-gradient(135deg,#1a1a2e 0%,#16213e 50%,#0f3460 100%);
                    padding:18px 24px;border-radius:8px;margin-bottom:12px;">
            <h2 style="color:#e0e0e0;margin:0;font-family:'Segoe UI',sans-serif;
                       font-weight:500;letter-spacing:0.5px;">
                Clinical reasoner
            </h2>
            <p style="color:#9a9abf;margin:4px 0 0 0;font-size:13px;">
                Select a patient, then explore via insight cards or free-text questions.
                All answers are deterministic and evidence-grounded.
            </p>
        </div>
        """))

        display(_r_ui)

    _progress_html.value = '<span style="font-size:12px;color:#4caf50;">Dashboard ready.</span>'
    _run_btn.disabled = False

_run_btn.on_click(_run_workflow)

display(widgets.VBox([
    widgets.HBox([_run_btn, _progress_html],
                  layout=widgets.Layout(align_items="center", gap="12px")),
    _main_out,
], layout=widgets.Layout(gap="8px")))


In [ ]:
# F. Diagnostics and deep dives (optional)

_diag_out = widgets.Output()

_dd_btns = {
    "Cross-fold stability": _deep_dive_a,
    "Inference health": _deep_dive_b,
    "Translational indices": _deep_dive_c,
    "Worked examples": _worked_examples,
}

def _make_dd_handler(fn, title):
    def handler(_b):
        with _diag_out:
            clear_output(wait=True)
            display(HTML(f'<h3 style="color:#333;margin-bottom:8px;">{title}</h3>'))
            try:
                fn()
            except Exception as e:
                print(f"Error running diagnostics: {e}")
    return handler

_btn_widgets = []
for label, fn in _dd_btns.items():
    b = widgets.Button(description=label, layout=widgets.Layout(width="180px"))
    b.on_click(_make_dd_handler(fn, label))
    _btn_widgets.append(b)

display(HTML("""
<details style="margin-top:16px;">
    <summary style="font-size:13px;color:#555;cursor:pointer;font-weight:600;">
        Diagnostics and reproducibility details</summary>
    <p style="padding:6px 0;font-size:12px;color:#666;">
        Click a button to run the corresponding diagnostic analysis.
        Each analysis produces tables and plots examining model validity.
    </p>
</details>
"""))

display(widgets.HBox(_btn_widgets, layout=widgets.Layout(gap="6px", margin="8px 0")))
display(_diag_out)


In [ ]:
# G. Reproducibility note
import datetime as _dt

display(HTML(f"""
<div style="background:#fafafa;border-top:1px solid #ddd;padding:12px 16px;
            margin-top:24px;font-size:11px;color:#777;line-height:1.6;">
    <strong>Reproducibility</strong><br>
    Run timestamp: {_dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}<br>
    Python: {sys.version.split()[0]} |
    NumPy: {np.__version__} |
    pandas: {pd.__version__} |
    matplotlib: {__import__("matplotlib").__version__}<br>
    Data bundle: {ROOT_PREFIX.rstrip("/") if ROOT_PREFIX else "(root)"} |
    Folds: {", ".join(FOLD_IDS)}<br>
    Outputs are rendered inline. No external files are written by this notebook.
</div>
"""))
